# Study of prediction & reconstruction performance 

## *Experiences description*

### Experiment 3 - Autoencoders Trained on Increasing Climate Diversity

In this third experiment, we want to study the distribution shift of our data across climates inside an autoencoder (AE). So we focus on the latent representation of an AE trained on some climates.

We retrieve the patchs used in the second experiment. We randomly split these samples in train/val/test datasets for each climate.
We then train the AE on one of these configurations (using the train and validation sets) :
- historical climate
- historical and ssp245 climates
- historical, ssp245 and ssp370 climates
- all climates

Then we evaluate the reconstruction quality of the AE on all climates (using the test sets)

We then project the latent representations of the test sets into a common PCA space, and we plot some visualization of this PCA space. 

And finally we perform the same analyzes between distributions then in the second experiment : 
- compute multivariate shift metrics in that latent PCA space;
- analyze the moments of the latent principal components;
- analyze extreme latent scores;
- analyze seasonal shifts in latent PCA space.

### Experiment 4 - Invariant Autoencoder with Latent Alignment

In this fourth experiment, we dive a step closer to the real CERA architecture by adding an explicit climate invariance term to the autoencoder loss.

The idea is to test wether adding an alignment loss between climates will effectivelly bring different climate distributions closer compared to the raw data and to the simple AE architecture. Here we only train the AE on the historical climate and on SSP245 to reproduce the CERA architecture (one "cold" and one "warm" climate).

Training set:
- **historical + ssp245**

Loss:
- reconstruction loss on all samples;
- alignment loss between latent samples from **historical** and **ssp245**.

Loss equation : 
$$
L = L_{\text{rec}} + \lambda_{\text{Align}} \cdot \text{Align}(Z^{\text{hist}}_{\text{align}}, Z^{\text{ssp245}}_{\text{align}})
$$

Alignment method : 

Here we consider two different method to align the historical and SSP245 climates ;
- We consider a sliced Wasserstein alignment loss 
- And a adversarial classifier.

Note that in CERA, the method used is Earth Mover's Distance (EMD), but EMD can be expensive in high dimension, it is why we use a sliced Wasserstein alignment loss, which is a practical EMD-style approximation.

Interpretation:
- decreasing the alignment term should reduce latent distribution shift between historical and ssp245, and potentially between historical and other warmer climates;
- this must be balanced against reconstruction quality;
- PCA visualizations help determine whether the latent clouds become more mixed and allow us to calculate the same metrics as before onto our "normalized" PCA space.

### Experiment 5 - CERA-like architecture

In this fifth experiment, we add a predictor to the architecture considered in the fourth experiment. We thus now consider : AE (constructed either with cnn2D or MLPs) (and with either sliced wasserstein distance alignment or adversarial classifier alignment) and a predictor using only the aligned part of the historical climate latent representations. The predictor needs to predict a given variable field over the whole grid of the samples. This architecture will be called a CERA-like architecture.

The same test/train/val split than before is used.

This CERA-like setup extends the invariant AE by adding a precipitation predictor:
- AE input: multivariate samples from historical + ssp245 climates.
- AE losses: reconstruction + latent alignment on the first 48 latent dimensions.
- Predictor: MLP on aligned latent dimensions (historical only) to predict a given variable over all 70 patch points.

Global loss used for AE update:
$$
L_{AE} = L_{rec} + \lambda_{align} L_{align} + \lambda_{pred} L_{pred}
$$

And we perform the predictor update at the same time, to be consistent with the end to end training used in the original CERA architecture.

## Part 0 - Global Configuration

**Library import**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
from IPython.display import display
import seaborn as sns
import json
from pathlib import Path
from sklearn.metrics import r2_score, mean_squared_error
from pathlib import Path
import pickle
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde
from matplotlib.colors import Normalize
from scipy.ndimage import gaussian_filter1d
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cfeature

**Data Loading**

In [ ]:
num_sample = 1000000
chosen_autoencoder_type = "CNN" # choose between "MLP" and "CNN"
inv_alignment_method = "swd" # choose between "swd" and "adversarial"
variable = "pr" # variable to predict
val_fraction = 0.05
test_fraction = 0.15
cera_lambda_align = 0.0001
cera_lambda_pred = 0.01

In [ ]:
precomputed_dir = Path(f"/glade/derecho/scratch/tsalin/CMIP/derived/multivariate_samples_optimized_NGS_v1000")
if not precomputed_dir.exists():
    raise FileNotFoundError(f"Precomputed data directory not found: {precomputed_dir}")

with open(precomputed_dir / "run_config.json", "r", encoding="utf-8") as f:
    run_cfg = json.load(f)

climate_order = list(run_cfg["climate_order"])
climate_colors = dict(run_cfg["climate_colors"])
selected_variables_full = list(run_cfg["selected_variables"])
max_abs_lat = float(run_cfg["max_abs_lat"])
patch_size_km = float(run_cfg["patch_size_km"])
time_stride = int(run_cfg["time_stride"])
max_samples_per_climate = int(run_cfg["max_samples_per_climate"])
random_seed = int(run_cfg["random_seed"])
n_lat = int(run_cfg["n_lat"])
n_lon = int(run_cfg["n_lon"])
grid_points_per_patch = int(run_cfg["grid_points_per_patch"])
n_patches = int(run_cfg["n_patches"])

We import the evaluation DataFrames

In [ ]:
evaluation_root = Path("/glade/work/tsalin/CMIP/model_evaluation")
if not evaluation_root.exists():
    raise FileNotFoundError(f"Model evaluation directory not found: {evaluation_root}")


def _load_quality_payload(file_path):
    """Load a quality payload dict from a pkl file."""
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"Model evaluation file not found: {file_path}")
    with open(file_path, "rb") as fh:
        content = pickle.load(fh)
    if not isinstance(content, dict):
        raise TypeError(f"Expected a dict payload in {file_path.name}, got {type(content)}")
    return content


def _load_history_df(file_path):
    """Load a history DataFrame from a pkl file."""
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"Model evaluation file not found: {file_path}")
    with open(file_path, "rb") as fh:
        content = pickle.load(fh)
    if not isinstance(content, pd.DataFrame):
        raise TypeError(f"Expected a DataFrame in {file_path.name}, got {type(content)}")
    return content.copy()


# ── exp5 CERA ─────────────────────────────────────────────────────────────────

_cera_prefix = f"cera_ns{num_sample}_{chosen_autoencoder_type}_{inv_alignment_method}_{variable}_{val_fraction}_{test_fraction}_{cera_lambda_align}_{cera_lambda_pred}_"
CERA_quality = evaluation_root / "CERA" / f"{_cera_prefix}quality_df.pkl"

CERA_history = evaluation_root / "CERA" / f"{_cera_prefix}history_df.pkl"

# ── exp5 CERA full latent ─────────────────────────────────────────────────────

_cera_fl_prefix      = f"cera_full_latent_ns{num_sample}_{chosen_autoencoder_type}_{inv_alignment_method}_{variable}_{val_fraction}_{test_fraction}_{cera_lambda_align}_{cera_lambda_pred}_"
CERA_full_latent_quality = evaluation_root / "CERA_full_latent" / f"{_cera_fl_prefix}quality_df.pkl"

CERA_full_latent_history = evaluation_root / "CERA_full_latent" / f"{_cera_fl_prefix}history_df.pkl"

# ── exp5 baseline simple ──────────────────────────────────────────────────────

_bs_prefix           = f"baseline_simple_ns{num_sample}_{variable}_{val_fraction}_{test_fraction}_"
Baseline_simple_quality = evaluation_root / "Baseline_simple" / f"{_bs_prefix}quality_df.pkl"

Baseline_simple_history = evaluation_root / "Baseline_simple" / f"{_bs_prefix}history_df.pkl"

# ── exp5 baseline physical ────────────────────────────────────────────────────

_bp_prefix              = f"baseline_physical_ns{num_sample}_{variable}_{val_fraction}_{test_fraction}_"
Baseline_physical_quality = evaluation_root / "Baseline_physical" / f"{_bp_prefix}quality_df.pkl"

Baseline_physical_history = evaluation_root / "Baseline_physical" / f"{_bp_prefix}history_df.pkl"


# ── exp5 baseline ClimaX ──────────────────────────────────────────────────────

_climax_prefix          = f"baseline_ClimaX_ns{num_sample}_{variable}_{val_fraction}_{test_fraction}_"
Baseline_ClimaX_quality = evaluation_root / "Baseline_ClimaX" / f"{_climax_prefix}quality_df.pkl"

Baseline_ClimaX_history = evaluation_root / "Baseline_ClimaX" / f"{_climax_prefix}history_df.pkl"


# ── exp5 baseline CERA noalign ────────────────────────────────────────────────

_noalign_prefix              = f"baseline_CERA_noalign_ns{num_sample}_{chosen_autoencoder_type}_{variable}_{val_fraction}_{test_fraction}_{cera_lambda_pred}_"
Baseline_CERA_noalign_quality = evaluation_root / "Baseline_CERA_noalign" / f"{_noalign_prefix}quality_df.pkl"

Baseline_CERA_noalign_history = evaluation_root / "Baseline_CERA_noalign" / f"{_noalign_prefix}history_df.pkl"

# ── exp4 ──────────────────────────────────────────────────────────────────────
_exp4_prefix        = f"exp4_ns{num_sample}_{chosen_autoencoder_type}_{inv_alignment_method}_{variable}_{val_fraction}_{test_fraction}_{cera_lambda_align}_"
exp4_reconstruction = evaluation_root / "exp4" / f"{_exp4_prefix}reconstruction_df.pkl"

# ── exp3 AEh  ───────
_exp3_prefix_AEh        = f"exp3_ns{num_sample}_{chosen_autoencoder_type}_AEh_{variable}_{val_fraction}_{test_fraction}_"
exp3_reconstruction_AEh = evaluation_root / "exp3" / f"{_exp3_prefix_AEh}reconstruction_df.pkl"

# ── exp3 AEhs2  ───────
_exp3_prefix_AEhs2        = f"exp3_ns{num_sample}_{chosen_autoencoder_type}_AEhs2_{variable}_{val_fraction}_{test_fraction}_"
exp3_reconstruction_AEhs2 = evaluation_root / "exp3" / f"{_exp3_prefix_AEhs2}reconstruction_df.pkl"

# ── exp3 AEhs2s3  ───────
_exp3_prefix_AEhs2s3        = f"exp3_ns{num_sample}_{chosen_autoencoder_type}_AEhs2s3_{variable}_{val_fraction}_{test_fraction}_"
exp3_reconstruction_AEhs2s3 = evaluation_root / "exp3" / f"{_exp3_prefix_AEhs2s3}reconstruction_df.pkl"

# ── exp3 AEall  ───────
_exp3_prefix_AEall        = f"exp3_ns{num_sample}_{chosen_autoencoder_type}_AEall_{variable}_{val_fraction}_{test_fraction}_"
exp3_reconstruction_AEall = evaluation_root / "exp3" / f"{_exp3_prefix_AEall}reconstruction_df.pkl"


# Load quality payloads
cera_quality_payload                  = _load_quality_payload(CERA_quality)
cera_full_latent_quality_payload      = _load_quality_payload(CERA_full_latent_quality)
baseline_simple_quality_payload       = _load_quality_payload(Baseline_simple_quality)
baseline_physical_quality_payload     = _load_quality_payload(Baseline_physical_quality)
baseline_climax_quality_payload       = _load_quality_payload(Baseline_ClimaX_quality)
baseline_cera_noalign_quality_payload = _load_quality_payload(Baseline_CERA_noalign_quality)

exp4_reconstruction_payload           = _load_quality_payload(exp4_reconstruction)
exp3_reconstruction_AEh_payload           = _load_quality_payload(exp3_reconstruction_AEh)
exp3_reconstruction_AEhs2_payload         = _load_quality_payload(exp3_reconstruction_AEhs2)
exp3_reconstruction_AEhs2s3_payload       = _load_quality_payload(exp3_reconstruction_AEhs2s3)
exp3_reconstruction_AEall_payload         = _load_quality_payload(exp3_reconstruction_AEall)


# Load history DataFrames
cera_history_df                  = _load_history_df(CERA_history)
cera_full_latent_history_df      = _load_history_df(CERA_full_latent_history)
baseline_simple_history_df       = _load_history_df(Baseline_simple_history)
baseline_physical_history_df     = _load_history_df(Baseline_physical_history)
baseline_climax_history_df       = _load_history_df(Baseline_ClimaX_history)
baseline_cera_noalign_history_df = _load_history_df(Baseline_CERA_noalign_history)

## Part 1 - Evaluation metrics computation

In [ ]:
def _safe_r2(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    if y_true.size < 2:
        return np.nan
    return float(r2_score(y_true, y_pred))


def _safe_rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    if y_true.size == 0:
        return np.nan
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def _parse_variable_slices(value_names):
    """Return list of (var_name, slice) in column order, one entry per unique variable."""
    slices = []
    seen = {}
    for i, name in enumerate(value_names):
        var = str(name).split("@")[0]
        if var not in seen:
            seen[var] = i
            slices.append((var, i))
    result = []
    for k, (var, start) in enumerate(slices):
        end = slices[k + 1][1] if k + 1 < len(slices) else len(value_names)
        result.append((var, slice(start, end)))
    return result


def _compute_scores_for_component(meta_df, truth_arr, pred_arr, value_names):
    """
    Vectorized score computation for one component (reconstruction or prediction).
    Returns a DataFrame with metadata + scores + truth/pred as numpy rows.
    Downstream visualization cells access truth_values/pred_values/value_names
    per row, so these columns are preserved as numpy arrays (not Python lists).
    """
    var_slices = _parse_variable_slices(value_names)
    N = len(meta_df)
    meta_reset = meta_df.reset_index(drop=True)

    # Per-sample R²: use per-climate column mean as baseline (avoids division by zero)
    sample_r2_by_var = {}
    for var, sl in var_slices:
        t = truth_arr[:, sl].astype(np.float64)
        p = pred_arr[:, sl].astype(np.float64)
        ss_res = np.nansum((t - p) ** 2, axis=1)
        t_mean_by_sample = np.empty_like(t)
        for scenario, grp in meta_reset.groupby("scenario", sort=False):
            idx = grp.index.to_numpy()
            t_mean_by_sample[idx] = np.nanmean(t[idx], axis=0, keepdims=True)
        ss_tot = np.nansum((t - t_mean_by_sample) ** 2, axis=1)
        with np.errstate(invalid="ignore", divide="ignore"):
            r2 = np.where(ss_tot > 0, 1.0 - ss_res / ss_tot, np.nan)
        sample_r2_by_var[var] = r2  # [N]

    var_names = [v for v, _ in var_slices]
    sample_r2_list = [
        {var: float(sample_r2_by_var[var][i]) for var in var_names}
        for i in range(N)
    ]

    # Global R²/RMSE per scenario (numpy slicing, no row iteration)
    global_scores = {}
    for scenario, grp in meta_reset.groupby("scenario", sort=False):
        idx = grp.index.to_numpy()
        r2g, rmseg = {}, {}
        for var, sl in var_slices:
            t_all = truth_arr[idx, sl].ravel().astype(np.float64)
            p_all = pred_arr[idx, sl].ravel().astype(np.float64)
            r2g[var]   = _safe_r2(t_all, p_all)
            rmseg[var] = _safe_rmse(t_all, p_all)
        global_scores[scenario] = {"r2_global": r2g, "rmse_global": rmseg}

    out = meta_reset.copy()
    out["r2"]          = sample_r2_list
    out["r2_global"]   = out["scenario"].map(lambda s: global_scores[s]["r2_global"])
    out["rmse_global"] = out["scenario"].map(lambda s: global_scores[s]["rmse_global"])
    # Store numpy rows (views into the original array — no data duplication)
    out["truth_values"] = list(truth_arr)
    out["pred_values"]  = list(pred_arr)
    out["value_names"]  = [list(value_names)] * N
    return out
def add_sample_and_global_scores(payload):
    """
    Takes a quality payload dict (new numpy format) and returns a complete DataFrame
    with per-sample and global R²/RMSE scores. The output is compatible with all
    downstream visualization cells that access truth_values/pred_values/value_names.
    """
    frames = []

    if "meta_reconstruction" in payload:
        frames.append(_compute_scores_for_component(
            payload["meta_reconstruction"],
            payload["truth_reconstruction"],
            payload["pred_reconstruction"],
            payload["reconstruction_value_names"],
        ))

    if "meta_prediction" in payload:
        frames.append(_compute_scores_for_component(
            payload["meta_prediction"],
            payload["truth_prediction"],
            payload["pred_prediction"],
            payload["prediction_value_names"],
        ))

    if not frames:
        raise ValueError("Payload has neither 'meta_reconstruction' nor 'meta_prediction'.")

    return pd.concat(frames, ignore_index=True)

In [ ]:
complete_cera_quality_df = add_sample_and_global_scores(cera_quality_payload)
del cera_quality_payload

complete_cera_full_latent_quality_df = add_sample_and_global_scores(cera_full_latent_quality_payload)
del cera_full_latent_quality_payload

complete_baseline_simple_quality_df = add_sample_and_global_scores(baseline_simple_quality_payload)
del baseline_simple_quality_payload

complete_baseline_physical_quality_df = add_sample_and_global_scores(baseline_physical_quality_payload)
del baseline_physical_quality_payload

complete_baseline_climax_quality_df = add_sample_and_global_scores(baseline_climax_quality_payload)
del baseline_climax_quality_payload

complete_baseline_cera_noalign_quality_df = add_sample_and_global_scores(baseline_cera_noalign_quality_payload)
del baseline_cera_noalign_quality_payload

complete_exp4_reconstruction_df = add_sample_and_global_scores(exp4_reconstruction_payload)
del exp4_reconstruction_payload

complete_exp3_AEh_reconstruction_df = add_sample_and_global_scores(exp3_reconstruction_AEh_payload)
del exp3_reconstruction_AEh_payload

complete_exp3_AEhs2_reconstruction_df = add_sample_and_global_scores(exp3_reconstruction_AEhs2_payload)
del exp3_reconstruction_AEhs2_payload

complete_exp3_AEhs2s3_reconstruction_df = add_sample_and_global_scores(exp3_reconstruction_AEhs2s3_payload)
del exp3_reconstruction_AEhs2s3_payload

complete_exp3_AEall_reconstruction_df = add_sample_and_global_scores(exp3_reconstruction_AEall_payload)
del exp3_reconstruction_AEall_payload


## Part 2 - Plotting quality results - reconstruction

In [ ]:
reconstruction_sources = [
    ("CERA", complete_cera_quality_df),
    ("Baseline no align", complete_baseline_cera_noalign_quality_df),
    ("ClimaX", complete_baseline_climax_quality_df),
    ("CERA full latent", complete_cera_full_latent_quality_df),
    ("Exp4", complete_exp4_reconstruction_df),
    ("Exp3 AEh", complete_exp3_AEh_reconstruction_df),
    ("Exp3 AEhs2", complete_exp3_AEhs2_reconstruction_df),
    ("Exp3 AEhs2s3", complete_exp3_AEhs2s3_reconstruction_df),
    ("Exp3 AEall", complete_exp3_AEall_reconstruction_df),
]

setup_order = [
    "CERA",
    "Baseline no align",
    "ClimaX",
    "CERA full latent",
    "Exp4",
    "Exp3 AEh",
    "Exp3 AEhs2",
    "Exp3 AEhs2s3",
    "Exp3 AEall"
]

In [ ]:
climates = ["historical", "ssp126", "ssp245", "ssp370", "ssp585"]

r2_rows = []

for setup_name, setup_df in reconstruction_sources:
    required_columns = {"component", "scenario", "r2_global"}
    missing_columns = required_columns - set(setup_df.columns)
    if missing_columns:
        raise KeyError(f"Missing columns {sorted(missing_columns)} in {setup_name} dataframe.")

    reconstruction_df = setup_df[setup_df["component"] == "reconstruction"].copy()
    if reconstruction_df.empty:
        continue

    for scenario_name, scenario_df in reconstruction_df.groupby("scenario", sort=False):
        climate = str(scenario_name)
        if climate not in climates:
            continue

        r2_global = scenario_df.iloc[0]["r2_global"]
        if not isinstance(r2_global, dict):
            raise TypeError(f"Expected r2_global to be a dictionary in {setup_name} / {scenario_name}.")

        for reconstructed_variable, r2_value in r2_global.items():
            reconstructed_variable = str(reconstructed_variable)
            if reconstructed_variable == variable:
                continue

            r2_rows.append({
                "setup": setup_name,
                "variable": reconstructed_variable,
                "climate": climate,
                "r2_global": r2_value,
            })

if not r2_rows:
    raise ValueError("No reconstructed variables found for the recap dataframe.")

r2_long_df = pd.DataFrame(r2_rows)

reconstruction_recap_df = (
    r2_long_df
    .groupby(["setup", "variable"], as_index=False)["r2_global"]
    .mean()
    .rename(columns={"r2_global": "r2_mean"})
)

reconstruction_recap_df["setup"] = pd.Categorical(
    reconstruction_recap_df["setup"], categories=setup_order, ordered=True
)
reconstruction_recap_df = reconstruction_recap_df.sort_values(["setup", "variable"]).reset_index(drop=True)

output_dir = Path("/glade/u/home/tsalin/CMIP/reportGraph")
output_dir.mkdir(parents=True, exist_ok=True)
reconstruction_recap_df.to_csv(output_dir / "reconstruction_recap_df.csv", index=False)

reconstruction_recap_df


In [ ]:
climate_offsets = {
    "historical": (0, 0),
    "ssp245": (0, 1),
    "ssp370": (1, 0),
    "ssp585": (1, 1),
}

r2_heatmap_rows = []
variable_order = []
seen_variables = set()

for setup_name, setup_df in reconstruction_sources:
    required_columns = {"component", "scenario", "r2_global"}
    missing_columns = required_columns - set(setup_df.columns)
    if missing_columns:
        raise KeyError(f"Missing columns {sorted(missing_columns)} in {setup_name} dataframe.")

    reconstruction_df = setup_df[setup_df["component"] == "reconstruction"].copy()
    if reconstruction_df.empty:
        continue

    for scenario_name, scenario_df in reconstruction_df.groupby("scenario", sort=False):
        climate = str(scenario_name)
        if climate not in climate_offsets:
            continue

        r2_global = scenario_df.iloc[0]["r2_global"]
        if not isinstance(r2_global, dict):
            raise TypeError(f"Expected r2_global to be a dictionary in {setup_name} / {scenario_name}.")

        for reconstructed_variable, r2_value in r2_global.items():
            reconstructed_variable = str(reconstructed_variable)
            if reconstructed_variable == variable:
                continue

            if reconstructed_variable not in seen_variables:
                seen_variables.add(reconstructed_variable)
                variable_order.append(reconstructed_variable)

            r2_heatmap_rows.append({
                "setup": setup_name,
                "variable": reconstructed_variable,
                "climate": climate,
                "r2_global": r2_value,
            })

if not variable_order:
    raise ValueError("No reconstructed variables found for the heatmap.")

heatmap_matrix = np.full((2 * len(variable_order), 2 * len(setup_order)), np.nan, dtype=float)
setup_index = {setup_name: idx for idx, setup_name in enumerate(setup_order)}
variable_index = {reconstructed_variable: idx for idx, reconstructed_variable in enumerate(variable_order)}

for row in r2_heatmap_rows:
    setup_idx = setup_index[row["setup"]]
    variable_idx = variable_index[row["variable"]]
    climate_row, climate_col = climate_offsets[row["climate"]]
    heatmap_matrix[2 * variable_idx + climate_row, 2 * setup_idx + climate_col] = row["r2_global"]

finite_values = heatmap_matrix[np.isfinite(heatmap_matrix)]
if finite_values.size == 0:
    raise ValueError("No finite r2_global values found for the reconstruction heatmap.")

cmap = plt.get_cmap("Blues").copy()
cmap.set_bad("#FFFFFF")

fig, ax = plt.subplots(
    figsize=(max(11, 1.45 * len(setup_order) + 3), max(7, 0.55 * len(variable_order) + 2.5)),
    constrained_layout=True,
)
im = ax.imshow(heatmap_matrix, aspect="auto", cmap=cmap, vmin=float(finite_values.min()), vmax=1.0, origin="upper")

ax.set_xticks(np.arange(len(setup_order)) * 2 + 0.5)
ax.set_xticklabels(setup_order, rotation=25, ha="right")
ax.set_yticks(np.arange(len(variable_order)) * 2 + 0.5)
ax.set_yticklabels(variable_order)

ax.set_xlabel("Setup")
ax.set_ylabel("Reconstructed variable")
ax.set_title(
    "Global $R^2$ for reconstruction\n"
    "Quadrants: historical (top-left), ssp245 (top-right), ssp370 (bottom-left), ssp585 (bottom-right)"
)

ax.set_xticks(np.arange(-0.5, heatmap_matrix.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-0.5, heatmap_matrix.shape[0], 1), minor=True)
ax.grid(which="minor", color="white", linewidth=1.0)
ax.tick_params(which="minor", bottom=False, left=False)

for setup_boundary in np.arange(2, heatmap_matrix.shape[1], 2):
    ax.axvline(setup_boundary - 0.5, color="#111827", linewidth=1.3)
for variable_boundary in np.arange(2, heatmap_matrix.shape[0], 2):
    ax.axhline(variable_boundary - 0.5, color="#111827", linewidth=1.3)

colorbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
colorbar.set_label("Global $R^2$")

plt.show()

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────────────
VIZ_SETUP       = "CERA full latent"   
VIZ_CLIMATE     = "historical" 
VIZ_VARIABLE    = "tas"
VIZ_SAMPLE_IDX  = 0         # sample index within the selected setup and climate
# ─────────────────────────────────────────────────────────────────────────────────────

# ── patch catalog loading ───────────────────────────────────────────────────────
_catalog_path = None
for _name in ("patch_catalog.pkl", "patches_catalog.pkl", "patch_catalog.csv"):
    _p = precomputed_dir / _name
    if _p.exists():
        _catalog_path = _p
        break

if _catalog_path is None:
    raise FileNotFoundError(
        f"patch_catalog not found in {precomputed_dir}. "
        f"Files present: {sorted(p.name for p in precomputed_dir.iterdir())}"
    )

if _catalog_path.suffix == ".pkl":
    with open(_catalog_path, "rb") as _f:
        _patch_catalog = pickle.load(_f)
else:
    _patch_catalog = pd.read_csv(_catalog_path)


def _get_patch_lat_lon_grids(patch_id):
    rows = _patch_catalog.loc[_patch_catalog["patch_id"] == int(patch_id)]
    if rows.empty:
        raise ValueError(f"patch_id={patch_id} not found in patch_catalog.")
    info = rows.iloc[0]
    local_n_lat = int(info["lat_stop_idx"] - info["lat_start_idx"])
    local_n_lon = int(info["lon_stop_idx"] - info["lon_start_idx"])
    lats = np.linspace(float(info["lat_start"]), float(info["lat_stop"]), local_n_lat)
    lons = np.linspace(float(info["lon_start"]), float(info["lon_stop"]), local_n_lon)
    lon_grid, lat_grid = np.meshgrid(lons, lats)
    return lat_grid, lon_grid, info


# ── DataFrame selection ────────────────────────────────────────────────────────────
_setup_dict = dict(reconstruction_sources)
if VIZ_SETUP not in _setup_dict:
    raise ValueError(f"Setup '{VIZ_SETUP}' unknown. Available: {list(_setup_dict.keys())}")

_src_df = _setup_dict[VIZ_SETUP]

_recon_df = _src_df[
    (_src_df["component"] == "reconstruction") &
    (_src_df["scenario"] == VIZ_CLIMATE)
].reset_index(drop=True)

if _recon_df.empty:
    raise ValueError(
        f"No reconstruction data for setup='{VIZ_SETUP}', climate='{VIZ_CLIMATE}'."
    )
if VIZ_SAMPLE_IDX >= len(_recon_df):
    raise IndexError(
        f"VIZ_SAMPLE_IDX={VIZ_SAMPLE_IDX} out of bounds (max {len(_recon_df) - 1})."
    )

_row = _recon_df.iloc[VIZ_SAMPLE_IDX]

if "patch_id" not in _row.index:
    raise KeyError(
        f"Column 'patch_id' missing from the DataFrame. "
        f"Available columns: {list(_recon_df.columns)}"
    )

# ── vizualised variable selection ─────────────────────────────────────────────
_vnames     = list(_row["value_names"])
_truth_flat = np.asarray(_row["truth_values"], dtype=float)
_recon_flat = np.asarray(_row["pred_values"],  dtype=float)

_all_vars   = sorted({str(n).split("@")[0] for n in _vnames})
_recon_vars = [v for v in _all_vars if v != variable] 

if VIZ_VARIABLE is None:
    VIZ_VARIABLE = _recon_vars[0] if _recon_vars else _all_vars[0]
elif VIZ_VARIABLE not in _all_vars:
    raise ValueError(f"Variable '{VIZ_VARIABLE}' absent. Available: {_all_vars}")

_vmask     = np.array([str(n).split("@")[0] == VIZ_VARIABLE for n in _vnames])
_truth_var = _truth_flat[_vmask]
_recon_var = _recon_flat[_vmask]

# ── Real coordinates via patch_catalog ─────────────────────────────────────────────
_patch_id = int(_row["patch_id"])
_lat_grid, _lon_grid, _patch_info = _get_patch_lat_lon_grids(_patch_id)

_n_lat_pp, _n_lon_pp = _lat_grid.shape
_truth_2d = _truth_var.reshape(_n_lat_pp, _n_lon_pp)
_recon_2d = _recon_var.reshape(_n_lat_pp, _n_lon_pp)

_lon_min = float(_lon_grid.min())
_lon_max = float(_lon_grid.max())
_lat_min = float(_lat_grid.min())
_lat_max = float(_lat_grid.max())

_margin_lon = max(1.0, 0.2 * (_lon_max - _lon_min))
_margin_lat = max(1.0, 0.2 * (_lat_max - _lat_min))
_extent_patch = [
    _lon_min - _margin_lon, _lon_max + _margin_lon,
    _lat_min - _margin_lat, _lat_max + _margin_lat,
]

_vmin = float(min(np.nanmin(_truth_2d), np.nanmin(_recon_2d)))
_vmax = float(max(np.nanmax(_truth_2d), np.nanmax(_recon_2d)))
_cmap = "YlOrRd"
_proj = ccrs.PlateCarree()

# ── plot : reconstruction | ground truth | world map ──────────────────────────────
fig = plt.figure(figsize=(17, 5.5), constrained_layout=True)
gs  = gridspec.GridSpec(1, 3, figure=fig, width_ratios=[2, 2, 1.4])


def _patch_panel(pos):
    ax = fig.add_subplot(pos, projection=_proj)
    ax.set_extent(_extent_patch, crs=_proj)
    ax.coastlines(resolution="110m", linewidth=0.8, color="black")
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, edgecolor="gray")
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.35)
    gl.top_labels   = False
    gl.right_labels = False
    return ax


ax_recon = _patch_panel(gs[0])
ax_truth = _patch_panel(gs[1])
ax_world = fig.add_subplot(gs[2], projection=_proj)

# ── reconstruction pannel ─────────────────────────────────────────────────────────────
_im = ax_recon.pcolormesh(
    _lon_grid, _lat_grid, _recon_2d,
    shading="auto", cmap=_cmap, vmin=_vmin, vmax=_vmax, transform=_proj,
)
ax_recon.scatter(
    _lon_grid.ravel(), _lat_grid.ravel(),
    s=8, color="black", alpha=0.4, transform=_proj, zorder=4,
)
ax_recon.set_title(
    f"Reconstruction  ({VIZ_SETUP})\n{VIZ_VARIABLE}  ·  {VIZ_CLIMATE}  ·  sample #{VIZ_SAMPLE_IDX}",
    fontweight="bold",
)

# ── ground truth pannel ───────────────────────────────────────────────────────────────
ax_truth.pcolormesh(
    _lon_grid, _lat_grid, _truth_2d,
    shading="auto", cmap=_cmap, vmin=_vmin, vmax=_vmax, transform=_proj,
)
ax_truth.scatter(
    _lon_grid.ravel(), _lat_grid.ravel(),
    s=8, color="black", alpha=0.4, transform=_proj, zorder=4,
)
ax_truth.set_title(
    f"Ground Truth\n{VIZ_VARIABLE}  ·  {VIZ_CLIMATE}  ·  sample #{VIZ_SAMPLE_IDX}",
    fontweight="bold",
)

# ── shared Colorbar ──────────────────────────────────────────────────────────────────
fig.colorbar(
    _im, ax=[ax_recon, ax_truth],
    orientation="vertical", fraction=0.035, pad=0.03,
    label=VIZ_VARIABLE,
)

# ── World map — geographical context ───────────────────────────────────────────────
ax_world.set_global()
ax_world.coastlines(resolution="110m", linewidth=0.7, color="black")
ax_world.add_feature(cfeature.BORDERS, linewidth=0.3, edgecolor="gray")
ax_world.gridlines(linewidth=0.3, alpha=0.35)

ax_world.add_patch(mpatches.Rectangle(
    (_lon_min, _lat_min), _lon_max - _lon_min, _lat_max - _lat_min,
    linewidth=2.0, edgecolor="red", facecolor="none",
    transform=_proj, zorder=5,
))

ax_world.set_title(
    f"Patch location\npatch_id = {_patch_id}",
    fontweight="bold",
)

fig.suptitle(
    f"Reconstruction vs Ground Truth  ·  {VIZ_SETUP}  ·  {VIZ_CLIMATE}  ·  {VIZ_VARIABLE}",
    fontweight="bold", fontsize=13,
)
plt.show()

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────────────
VIZ_CLIMATE_LAT  = "historical"
VIZ_VARIABLE_LAT =  "tas"     
N_LAT_BINS       = 30      
# ─────────────────────────────────────────────────────────────────────────────────────


# ── Patch catalog  ─────────────────────────────────────────────
if "_patch_catalog" not in dir():
    _cat_path_lat = None
    for _n in ("patch_catalog.pkl", "patches_catalog.pkl", "patch_catalog.csv"):
        _pp = precomputed_dir / _n
        if _pp.exists():
            _cat_path_lat = _pp
            break
    if _cat_path_lat is None:
        raise FileNotFoundError(f"patch_catalog not found in {precomputed_dir}.")
    with open(_cat_path_lat, "rb") as _f:
        _patch_catalog = pickle.load(_f)

_cat_idx = _patch_catalog.set_index("patch_id")

# ──────────────────────────────────────────────────────
_sample_vars_lat = None
for _sn_l, _sdf_l in reconstruction_sources:
    _tmp_l = _sdf_l[
        (_sdf_l["component"] == "reconstruction") &
        (_sdf_l["scenario"] == VIZ_CLIMATE_LAT)
    ]
    if not _tmp_l.empty and "r2" in _tmp_l.columns:
        _first_r2 = _tmp_l.iloc[0]["r2"]
        if isinstance(_first_r2, dict):
            _sample_vars_lat = sorted(_first_r2.keys())
            break

if _sample_vars_lat is None:
    raise ValueError(
        f"No reconstruction data with column 'r2' for climate='{VIZ_CLIMATE_LAT}'."
    )

_recon_vars_lat = [v for v in _sample_vars_lat if v != variable]
if VIZ_VARIABLE_LAT is None:
    VIZ_VARIABLE_LAT = _recon_vars_lat[0] if _recon_vars_lat else _sample_vars_lat[0]
elif VIZ_VARIABLE_LAT not in _sample_vars_lat:
    raise ValueError(f"Variable '{VIZ_VARIABLE_LAT}' absent. Available: {_sample_vars_lat}")

# ── Collect of R² by latitude ───────────────────────────────────
# For each sample :
#   - latitude = middle of the patch (lat_start + lat_stop) / 2 via patch_catalog
#   - R²       = r2[VIZ_VARIABLE_LAT] pre computed

_lat_r2_by_setup = {}

for _sname, _sdf in reconstruction_sources:
    _filt = _sdf[
        (_sdf["component"] == "reconstruction") &
        (_sdf["scenario"] == VIZ_CLIMATE_LAT)
    ]
    if _filt.empty or "patch_id" not in _filt.columns or "r2" not in _filt.columns:
        continue

    _pairs = []

    for _, _row in _filt.iterrows():
        _pid = int(_row["patch_id"])
        if _pid not in _cat_idx.index:
            continue

        _info      = _cat_idx.loc[_pid]
        _lat_c_val = (float(_info["lat_start"]) + float(_info["lat_stop"])) / 2.0

        _r2_dict = _row["r2"]
        if not isinstance(_r2_dict, dict):
            continue
        _r2_val = _r2_dict.get(VIZ_VARIABLE_LAT, np.nan)
        if np.isfinite(float(_r2_val)):
            _pairs.append((_lat_c_val, float(_r2_val)))

    if _pairs:
        _lat_r2_by_setup[_sname] = _pairs

if not _lat_r2_by_setup:
    raise ValueError(
        "No R² data available by latitude. "
        "Check that patch_id and r2 are present in the DataFrames."
    )

# ── Bins of same latitude ──────────────────────────────────────────────────────────
_all_lats_flat  = [lat for pairs in _lat_r2_by_setup.values() for lat, _ in pairs]
_lat_global_min = min(_all_lats_flat)
_lat_global_max = max(_all_lats_flat)
_lat_bins_plot  = np.linspace(_lat_global_min, _lat_global_max, N_LAT_BINS + 1)
_lat_centers_p  = (_lat_bins_plot[:-1] + _lat_bins_plot[1:]) / 2

# ── colors ────────────────────────────────────────────────────────────────────────────
_setup_colors_lat = {
    "CERA":              "#2F3B52",
    "Baseline no align": "#2C7FB8",
    "ClimaX":             "#A6CEE3",
    "CERA full latent":   "#E5C494",
    "Exp4":              "#F1A340",
    "exp3 AEh":          "#E63946",
    "exp3 AEhs2":        "#F4A261",
    "exp3 AEhs2s3":      "#2A9D8F",
    "exp3 AEall":        "#6A3D9A",
}
_default_colors_lat = plt.rcParams["axes.prop_cycle"].by_key()["color"]

# ── plot ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)

for _ci, (_sname, _pairs) in enumerate(_lat_r2_by_setup.items()):
    _lats_a = np.array([p[0] for p in _pairs])
    _r2_a   = np.array([p[1] for p in _pairs])
    _color  = _setup_colors_lat.get(_sname, _default_colors_lat[_ci % len(_default_colors_lat)])

    _mean_r2 = np.full(N_LAT_BINS, np.nan)
    _std_r2  = np.full(N_LAT_BINS, np.nan)

    for _bi in range(N_LAT_BINS):
        _mask_b = (_lats_a >= _lat_bins_plot[_bi]) & (_lats_a < _lat_bins_plot[_bi + 1])
        _n_b    = int(np.sum(_mask_b))
        if _n_b >= 2:
            _mean_r2[_bi] = np.mean(_r2_a[_mask_b])
            _std_r2[_bi]  = np.std(_r2_a[_mask_b], ddof=1)
        elif _n_b == 1:
            _mean_r2[_bi] = _r2_a[_mask_b][0]

    _valid = np.isfinite(_mean_r2)
    _lc    = _lat_centers_p[_valid]
    _mc    = _mean_r2[_valid]
    _sc    = np.where(np.isfinite(_std_r2[_valid]), _std_r2[_valid], 0.0)

    ax.plot(_lc, _mc, linewidth=1.8, color=_color, label=_sname)
    ax.fill_between(_lc, _mc - _sc, _mc + _sc, alpha=0.15, color=_color)

ax.axhline(0.0, color="#6B7280", linewidth=0.9, linestyle="--", alpha=0.7)
ax.set_xlabel("Latitude (°)", fontweight="bold")
ax.set_ylabel(f"$R^2$ ({VIZ_VARIABLE_LAT})", fontweight="bold")
ax.set_title(
    f"$R^2$ per latitude  ·  {VIZ_VARIABLE_LAT}  ·  {VIZ_CLIMATE_LAT}\n"
    "(Mean ± standard deviation, latitude = middle of the patch)",
    fontweight="bold",
)
ax.legend(title="Setup", fontsize=9, loc="best", framealpha=0.9)
ax.grid(alpha=0.25, linestyle=":")
plt.show()

## Part 3 - Plotting quality results - prediction

Definition of studied setups

In [ ]:
prediction_plot_sources_list = [
    ("CERA", complete_cera_quality_df),
    ("ClimaX", complete_baseline_climax_quality_df),
    ("Baseline no align", complete_baseline_cera_noalign_quality_df),
    ("CERA full latent", complete_cera_full_latent_quality_df),
    ("Baseline physical", complete_baseline_physical_quality_df),
    ("Baseline simple", complete_baseline_simple_quality_df),
]

prediction_plot_sources_reverse_list = [
    (complete_cera_quality_df, "CERA"),
    (complete_baseline_simple_quality_df, "Baseline Simple"),
    (complete_baseline_physical_quality_df, "Baseline Physical"),
    (complete_baseline_climax_quality_df, "Baseline ClimaX"),
    (complete_baseline_cera_noalign_quality_df, "Baseline CERA (no align)"),
    (complete_cera_full_latent_quality_df, "CERA Full Latent")
]

prediction_plot_sources_dict = {
    "CERA":                    complete_cera_quality_df,
    "Baseline no align":       complete_baseline_cera_noalign_quality_df,
    "ClimaX":                   complete_baseline_climax_quality_df,
    "Baseline simple":         complete_baseline_simple_quality_df,
    "Baseline physical":       complete_baseline_physical_quality_df,
    "CERA full latent":        complete_cera_full_latent_quality_df,
}

setup_order = ["CERA", "ClimaX", "Baseline no align", "CERA full latent", "Baseline physical", "Baseline simple"]

setup_markers = {
    "CERA": "o",
    "ClimaX": "s",
    "Baseline no align": "D",
    "CERA full latent": "^",
    "Baseline physical": "v",
    "Baseline simple": "P"
}

setup_colors = {
    "CERA": "#2F3B52",
    "ClimaX": "#A6CEE3",
    "Baseline no align": "#2C7FB8",
    "CERA full latent": "#E5C494",
    "Baseline physical": "#F1A340",
    "Baseline simple": "#E63946"
}

setups_info_summary = [
    ('complete_cera_quality_df', 'CERA', 'green', '-'),
    ('complete_cera_full_latent_quality_df', 'CERA full latent', 'blue', '-'),
    ('complete_baseline_simple_quality_df', 'Baseline Simple', 'violet', '-'),
    ('complete_baseline_physical_quality_df', 'Baseline Physical', 'red', '-'),
    ('complete_baseline_climax_quality_df', 'Baseline ClimaX', 'cyan', '-'),
    ('complete_baseline_cera_noalign_quality_df', 'Baseline CERA (no align)', 'orange', '-'),
]

R²

In [ ]:
r2_plot_sources = prediction_plot_sources_list

r2_plot_rows = []
for setup_name, df in r2_plot_sources:
    if "r2_global" not in df.columns:
        raise KeyError(f"Missing 'r2_global' column in {setup_name} dataframe.")

    for _, row in df.iterrows():
        r2_value = row["r2_global"]
        if isinstance(r2_value, dict):
            r2_value = r2_value.get(variable, np.nan)

        r2_plot_rows.append({
            "setup": setup_name,
            "scenario": row["scenario"],
            "r2_global": r2_value,
        })

r2_plot_df = pd.DataFrame(r2_plot_rows)
if r2_plot_df.empty:
    raise ValueError("No rows available to plot r2_global.")

scenario_order = list(dict.fromkeys(r2_plot_df["scenario"].tolist()))

fig, ax = plt.subplots(figsize=(4, 4.0), constrained_layout=True)
for setup_name in setup_order:
    setup_df = r2_plot_df[r2_plot_df["setup"] == setup_name].copy()
    if setup_df.empty:
        continue

    setup_df["scenario"] = pd.Categorical(setup_df["scenario"], categories=scenario_order, ordered=True)
    setup_df = setup_df.sort_values("scenario")
    ax.plot(
        setup_df["scenario"],
        setup_df["r2_global"],
        linestyle="None",
        marker=setup_markers[setup_name],
        markersize=7,
        color=setup_colors[setup_name],
        label=setup_name,
    )

ax.axhline(0.0, color="#6B7280", linewidth=1.0, linestyle="", alpha=0.8)
ax.set_xlabel("Scenario")
ax.set_ylabel(f"Global $R^2$ for {variable}")
ax.set_title(f"Global $R^2$ by scenario and setup for {variable}")
ax.grid(axis="y", alpha=0.25)
ax.set_ylim(-0.5, 1.2)

legend_handles = [
    Line2D([0], [0], color=setup_colors[setup_name], marker=setup_markers[setup_name], linewidth=0.0, label=setup_name)
    for setup_name in setup_order
]
ax.legend(handles=legend_handles, title="Setup")

plt.xticks(rotation=20, ha="right")
plt.show()

# ── Delta R² = R²(ssp) − R²(historical) ──────────────────────────────────────────────

# A single global R² per (setup, scenario) — r2_global is constant within each group
_r2_unique = (
    r2_plot_df
    .groupby(["setup", "scenario"], sort=False)["r2_global"]
    .first()
    .reset_index()
)
_r2_pivot = _r2_unique.pivot(index="setup", columns="scenario", values="r2_global")

# The reference scenario may be named "historical" or "hist" depending on the data
_ref_col = next((c for c in ("historical", "hist") if c in _r2_pivot.columns), None)
if _ref_col is None:
    raise ValueError(
        f"Historical scenario not found. Available scenarios: {list(_r2_pivot.columns)}"
    )

_ssp_cols = [c for c in _r2_pivot.columns if c != _ref_col]
if not _ssp_cols:
    raise ValueError("No SSP scenario found in the data.")

_delta_df = _r2_pivot[_ssp_cols].subtract(_r2_pivot[_ref_col], axis=0)

fig2, ax2 = plt.subplots(figsize=(4, 4.0), constrained_layout=True)

for setup_name in setup_order:
    if setup_name not in _delta_df.index:
        continue
    _row_delta = _delta_df.loc[setup_name]
    ax2.plot(
        _row_delta.index,
        _row_delta.values,
        linestyle="None",
        marker=setup_markers[setup_name],
        markersize=7,
        color=setup_colors[setup_name],
        label=setup_name,
    )

ax2.axhline(0.0, color="#6B7280", linewidth=1.0, linestyle="--", alpha=0.7)
ax2.set_xlabel("Scenario")
ax2.set_ylabel(f"Δ Global $R^2$ for {variable}")
ax2.set_title(f"Δ Global $R^2$ vs. {_ref_col} for {variable}")
ax2.grid(axis="y", alpha=0.25)
ax2.set_ylim(-2, 0.1)

_legend_handles2 = [
    Line2D([0], [0], color=setup_colors[s], marker=setup_markers[s], linewidth=0.0, label=s)
    for s in setup_order if s in _delta_df.index
]
ax2.legend(handles=_legend_handles2, title="Setup")

plt.xticks(rotation=20, ha="right")
plt.show()


RMSE

In [ ]:
rmse_plot_rows = []
for setup_name, df in r2_plot_sources:
    if "rmse_global" not in df.columns:
        raise KeyError(f"Missing 'rmse_global' column in {setup_name} dataframe.")

    for _, row in df.iterrows():
        rmse_value = row["rmse_global"]
        if isinstance(rmse_value, dict):
            rmse_value = rmse_value.get(variable, np.nan)

        rmse_plot_rows.append({
            "setup": setup_name,
            "scenario": row["scenario"],
            "rmse_global": rmse_value,
        })

rmse_plot_df = pd.DataFrame(rmse_plot_rows)
if rmse_plot_df.empty:
    raise ValueError("No rows available to plot rmse_global.")

scenario_order = list(dict.fromkeys(rmse_plot_df["scenario"].tolist()))

fig, ax = plt.subplots(figsize=(4, 4.0), constrained_layout=True)
for setup_name in setup_order:
    setup_df = rmse_plot_df[rmse_plot_df["setup"] == setup_name].copy()
    if setup_df.empty:
        continue

    setup_df["scenario"] = pd.Categorical(setup_df["scenario"], categories=scenario_order, ordered=True)
    setup_df = setup_df.sort_values("scenario")
    ax.plot(
        setup_df["scenario"],
        setup_df["rmse_global"],
        linestyle="None",
        marker=setup_markers[setup_name],
        markersize=7,
        color=setup_colors[setup_name],
        label=setup_name,
    )

ax.set_xlabel("Scenario")
ax.set_ylabel(f"Global RMSE for {variable}")
ax.set_title(f"Global RMSE by scenario and setup for {variable}")
ax.grid(axis="y", alpha=0.25)

legend_handles = [
    Line2D([0], [0], color=setup_colors[setup_name], marker=setup_markers[setup_name], linewidth=0.0, label=setup_name)
    for setup_name in setup_order
]
ax.legend(handles=legend_handles, title="Setup")

plt.xticks(rotation=20, ha="right")
plt.show()

# ── Ratio RMSE = RMSE(ssp) / RMSE(historical) ────────────────────────────────────────

_rmse_unique = (
    rmse_plot_df
    .groupby(["setup", "scenario"], sort=False)["rmse_global"]
    .first()
    .reset_index()
)
_rmse_pivot = _rmse_unique.pivot(index="setup", columns="scenario", values="rmse_global")

_ref_col = next((c for c in ("historical", "hist") if c in _rmse_pivot.columns), None)
if _ref_col is None:
    raise ValueError(
        f"No historical scenario found. Available scenarios: {list(_rmse_pivot.columns)}"
    )

_ssp_cols = [c for c in _rmse_pivot.columns if c != _ref_col]
if not _ssp_cols:
    raise ValueError("No SSP scenario found in the data.")

_ratio_df = _rmse_pivot[_ssp_cols].divide(_rmse_pivot[_ref_col], axis=0)

fig2, ax2 = plt.subplots(figsize=(4, 4.0), constrained_layout=True)

for setup_name in setup_order:
    if setup_name not in _ratio_df.index:
        continue
    _row_ratio = _ratio_df.loc[setup_name]
    ax2.plot(
        _row_ratio.index,
        _row_ratio.values,
        linestyle="None",
        marker=setup_markers[setup_name],
        markersize=7,
        color=setup_colors[setup_name],
        label=setup_name,
    )

ax2.axhline(1.0, color="#6B7280", linewidth=1.0, linestyle="--", alpha=0.7)
ax2.set_xlabel("Scenario")
ax2.set_ylabel(f"RMSE ratio vs. {_ref_col}\n({variable})")
ax2.set_title(f"RMSE(ssp) / RMSE({_ref_col})\n{variable}")
ax2.grid(axis="y", alpha=0.25)

_legend_handles2 = [
    Line2D([0], [0], color=setup_colors[s], marker=setup_markers[s], linewidth=0.0, label=s)
    for s in setup_order if s in _ratio_df.index
]
ax2.legend(handles=_legend_handles2, title="Setup")

plt.xticks(rotation=20, ha="right")
plt.show()


R² distribution

In [ ]:
# r2 distribution per sample (boxplot or violin plot)  


_violin_sources  = prediction_plot_sources_list
_setup_order_v   = setup_order
_violin_colors   = setup_colors
_climate_order_v = climate_order

# ── Collecte des R² par sample et R² global ──────────────────────────────────────────
_samples_v = {c: {s: []       for s in _setup_order_v} for c in _climate_order_v}
_r2g_v     = {c: {s: np.nan   for s in _setup_order_v} for c in _climate_order_v}

for _sname, _sdf in _violin_sources:
    _pred = _sdf[_sdf["component"] == "prediction"]
    if _pred.empty or "r2" not in _pred.columns:
        continue

    for _clim_raw, _grp in _pred.groupby("scenario", sort=False):
        _clim = str(_clim_raw)
        if _clim not in _climate_order_v:
            continue

        # r2_global
        if "r2_global" in _grp.columns:
            _rg = _grp.iloc[0]["r2_global"]
            if isinstance(_rg, dict):
                _rg = _rg.get(variable, np.nan)
            if np.isfinite(float(_rg)):
                _r2g_v[_clim][_sname] = float(_rg)

        # r2 per sample
        for _, _row in _grp.iterrows():
            _rs = _row["r2"]
            if isinstance(_rs, dict):
                _rs = _rs.get(variable, np.nan)
            if np.isfinite(float(_rs)):
                _samples_v[_clim][_sname].append(float(_rs))

# ── Plotting ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(16, 5), constrained_layout=True, sharey=True)

for _ci, _clim in enumerate(_climate_order_v):
    ax = axes[_ci]

    _pos, _data, _globals, _labels, _cols = [], [], [], [], []
    for _i, _sname in enumerate(_setup_order_v):
        _s = _samples_v[_clim][_sname]
        if not _s:
            continue
        _pos.append(_i + 1)
        _data.append(_s)
        _globals.append(_r2g_v[_clim][_sname])
        _labels.append(_sname)
        _cols.append(_violin_colors[_sname])

    if not _data:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes, color="grey")
        ax.set_title(_clim.upper(), fontweight="bold")
        continue

    _vp = ax.violinplot(_data, positions=_pos, showmedians=True, showextrema=True)

    for _vi, _body in enumerate(_vp["bodies"]):
        _body.set_facecolor(_cols[_vi])
        _body.set_alpha(0.6)
        _body.set_edgecolor(_cols[_vi])
    for _part in ("cmins", "cmaxes", "cbars"):
        if _part in _vp:
            _vp[_part].set_color("grey")
            _vp[_part].set_linewidth(0.8)
    _vp["cmedians"].set_color("white")
    _vp["cmedians"].set_linewidth(2.0)
    _vp["cmedians"].set_zorder(4)

    # R² global en losange rouge
    for _vi, (_p, _rg) in enumerate(zip(_pos, _globals)):
        if np.isfinite(_rg):
            ax.scatter(
                [_p], [_rg],
                marker="D", s=55, color="red", zorder=5,
                label="global $R^2$" if (_ci == 0 and _vi == 0) else "_nolegend_",
            )

    ax.axhline(0.0, color="#6B7280", linewidth=0.8, linestyle="--", alpha=0.6)
    ax.set_xticks(_pos)
    ax.set_xticklabels(_labels, rotation=20, ha="right", fontsize=9)
    ax.set_title(_clim.upper(), fontweight="bold")
    ax.grid(axis="y", alpha=0.25, linestyle=":")

axes[0].set_ylabel(f"$R^2$ ({variable})", fontweight="bold")

axes[-1].legend(
    handles=[Line2D([0], [0], marker="D", color="red", linewidth=0, markersize=6, label="global $R^2$")],
    loc="lower right", fontsize=9,
)

fig.suptitle(
    f"Distribution of prediction $R^2$ by setup and by climate  ·  {variable}",
    fontweight="bold", fontsize=13,
)
plt.show()

Observed vs Predicted Values (Scatter Plot)

In [ ]:
MAX_SCATTER_SAMPLES = 25000


def _extract_prediction_pairs(df, climate_name, max_samples=MAX_SCATTER_SAMPLES, seed=0):
    subset = df[(df["component"] == "prediction") & (df["scenario"] == climate_name)]
    truth_values = []
    pred_values = []

    if subset.empty:
        return np.array([]), np.array([])

    if "truth_values" not in subset.columns or "pred_values" not in subset.columns:
        return np.array([]), np.array([])

    for _, row in subset.iterrows():
        try:
            truth_array = np.asarray(row["truth_values"], dtype=float).ravel()
            pred_array = np.asarray(row["pred_values"], dtype=float).ravel()
        except Exception:
            continue

        if truth_array.size == 0 or pred_array.size == 0:
            continue

        pair_mask = np.isfinite(truth_array) & np.isfinite(pred_array)
        if not np.any(pair_mask):
            continue

        truth_sample = truth_array[pair_mask]
        pred_sample = pred_array[pair_mask]

        truth_values.append(float(np.nanmean(truth_sample)))
        pred_values.append(float(np.nanmean(pred_sample)))

    if len(truth_values) == 0:
        return np.array([]), np.array([])

    truth_values = np.asarray(truth_values, dtype=float)
    pred_values = np.asarray(pred_values, dtype=float)

    if truth_values.size > max_samples:
        rng = np.random.default_rng(seed)
        sample_indices = rng.choice(truth_values.size, size=max_samples, replace=False)
        truth_values = truth_values[sample_indices]
        pred_values = pred_values[sample_indices]

    return truth_values, pred_values


climates = complete_cera_quality_df["scenario"].unique().tolist()

setups_info = prediction_plot_sources_reverse_list

all_data = {}
global_t_min = np.inf
global_t_max = -np.inf
global_p_min = np.inf
global_p_max = -np.inf
global_z_min = np.inf
global_z_max = -np.inf

for climate_idx, climate_name in enumerate(climates):
    for setup_idx, (setup_df, setup_title) in enumerate(setups_info):
        if setup_df is None or setup_df.empty:
            continue

        truth_values, pred_values = _extract_prediction_pairs(setup_df, climate_name)
        if truth_values.size == 0:
            continue

        global_t_min = min(global_t_min, float(np.nanmin(truth_values)))
        global_t_max = max(global_t_max, float(np.nanmax(truth_values)))
        global_p_min = min(global_p_min, float(np.nanmin(pred_values)))
        global_p_max = max(global_p_max, float(np.nanmax(pred_values)))

        density = None
        if truth_values.size >= 3 and np.unique(np.column_stack([truth_values, pred_values]), axis=0).shape[0] >= 3:
            try:
                xy = np.vstack([truth_values, pred_values])
                density = gaussian_kde(xy)(xy)
                global_z_min = min(global_z_min, float(np.nanmin(density)))
                global_z_max = max(global_z_max, float(np.nanmax(density)))
            except Exception:
                density = None

        rmse_global = np.nan
        prediction_subset = setup_df[(setup_df["component"] == "prediction") & (setup_df["scenario"] == climate_name)]
        if not prediction_subset.empty and "rmse_global" in prediction_subset.columns:
            first_rmse = prediction_subset.iloc[0]["rmse_global"]
            if isinstance(first_rmse, dict):
                rmse_global = first_rmse.get(variable, np.nan)
            else:
                rmse_global = first_rmse

        all_data[(climate_idx, setup_idx)] = {
            "truth_values": truth_values,
            "pred_values": pred_values,
            "density": density,
            "rmse_global": rmse_global,
        }

if not np.isfinite(global_t_min) or not np.isfinite(global_t_max):
    raise ValueError("No finite truth values available for the prediction scatter plots.")
if not np.isfinite(global_p_min) or not np.isfinite(global_p_max):
    raise ValueError("No finite predicted values available for the prediction scatter plots.")

x_pad = 0.05 * (global_t_max - global_t_min) if global_t_max > global_t_min else 1.0
y_pad = 0.05 * (global_p_max - global_p_min) if global_p_max > global_p_min else 1.0
global_t_min -= x_pad
global_t_max += x_pad
global_p_min -= y_pad
global_p_max += y_pad

global_t_max = min(global_t_max, 40)   # x max → 40
global_p_max = min(global_p_max, 100)  # y max → 100


if np.isfinite(global_z_min) and np.isfinite(global_z_max):
    density_norm = Normalize(vmin=global_z_min, vmax=global_z_max)
else:
    density_norm = None

fig, axes = plt.subplots(5, 6, figsize=(18, 16), constrained_layout=False)

for climate_idx, climate_name in enumerate(climates):
    for setup_idx, (_, setup_title) in enumerate(setups_info):
        ax = axes[climate_idx, setup_idx]
        subplot_data = all_data.get((climate_idx, setup_idx))

        if subplot_data is None:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", fontsize=10)
            ax.set_title(setup_title, fontsize=11, fontweight="bold")
            ax.set_xlim(global_t_min, global_t_max)
            ax.set_ylim(global_p_min, global_p_max)
            ax.grid(alpha=0.25, linestyle=":")
            continue

        truth_values = subplot_data["truth_values"]
        pred_values = subplot_data["pred_values"]
        density = subplot_data["density"]
        rmse_global = subplot_data["rmse_global"]

        if truth_values.size == 0:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", fontsize=10)
            ax.set_title(setup_title, fontsize=11, fontweight="bold")
            ax.set_xlim(global_t_min, global_t_max)
            ax.set_ylim(global_p_min, global_p_max)
            ax.grid(alpha=0.25, linestyle=":")
            continue

        if density is not None and density_norm is not None:
            ax.scatter(
                truth_values,
                pred_values,
                c=density,
                s=16,
                cmap="hot",
                alpha=0.65,
                edgecolors="none",
                norm=density_norm,
            )
        else:
            ax.scatter(
                truth_values,
                pred_values,
                s=16,
                c="#6B7280",
                alpha=0.55,
                edgecolors="none",
            )

        ax.plot([global_t_min, global_t_max], [global_t_min, global_t_max], color="black", linestyle="--", linewidth=1.2, alpha=0.7)
        ax.set_xlim(global_t_min, global_t_max)
        ax.set_ylim(global_p_min, global_p_max)
        ax.set_xlabel("True value")
        ax.set_ylabel("Predicted value")
        ax.set_title(f"{setup_title}\nRMSE global = {rmse_global:.5f}", fontsize=11, fontweight="bold")
        ax.grid(alpha=0.25, linestyle=":")

    axes[climate_idx, 0].text(
        -0.35,
        0.5,
        climate_name.upper(),
        transform=axes[climate_idx, 0].transAxes,
        fontsize=13,
        fontweight="bold",
        ha="center",
        va="center",
        rotation=90,
    )

if density_norm is not None:
    plt.subplots_adjust(left=0.1, right=0.88, top=0.93, bottom=0.08, wspace=0.3, hspace=0.38)
    cbar_ax = fig.add_axes([0.90, 0.15, 0.018, 0.7])
    cbar = fig.colorbar(plt.cm.ScalarMappable(norm=density_norm, cmap="hot"), cax=cbar_ax)
    cbar.set_label("Density (global scale)", fontsize=11)
else:
    plt.subplots_adjust(left=0.1, right=0.95, top=0.93, bottom=0.08, wspace=0.3, hspace=0.38)

plt.suptitle(f"Prediction scatter plots for {variable} - True vs Predicted values", fontsize=15, fontweight="bold", y=0.995)
plt.show()

Latitudinal Prediction Profiles

In [ ]:
_UNIT_LABELS = {
    "tas": "tas (K)", "huss": "huss (kg/kg)", "pr": "pr (mm/day)",
    "psl": "psl (Pa)", "rsds": "rsds (W/m²)", "sfcWind": "sfcWind (m/s)",
    "ta": "ta (K)", "hus": "hus (kg/kg)", "ua": "ua (m/s)",
    "va": "va (m/s)", "wap": "wap (Pa/s)", "zg": "zg (m)",
}
_ylabel = _UNIT_LABELS.get(variable, variable)
# Profile of observed vs predicted variable by latitude
# Separate plots for each climate in a 2x2 grid

# Get dimensions and patch size
n_lat_per_patch = 10
n_lon_per_patch = 7
lat_edges = np.linspace(-max_abs_lat, max_abs_lat, n_lat + 1)
lat_centers = (lat_edges[:-1] + lat_edges[1:]) / 2.0
lat_min, lat_max = lat_centers.min(), lat_centers.max()

# Create higher-resolution latitude bins
n_lat_bins_hires = 30
lat_bins_hires = np.linspace(lat_min, lat_max, n_lat_bins_hires + 1)
lat_centers_hires = (lat_bins_hires[:-1] + lat_bins_hires[1:]) / 2.0

# Climate order
climates = complete_cera_quality_df["scenario"].unique().tolist()

# Create 2x2 figure
fig, axes = plt.subplots(3, 2, figsize=(16, 12), constrained_layout=True)
axes_flat = axes.flatten()

# Setup configurations: (varname, label, line_color, line_style)
setups_info = setups_info_summary


# For each climate, create a subplot
for climate_idx, climate in enumerate(climates):
    ax = axes_flat[climate_idx]
    
    # Store truth profile for this climate to plot once
    truth_valid_all = None
    lat_valid_all = None
    
    # For each setup
    for varname, label, color, linestyle in setups_info:
        if varname not in globals() or globals()[varname] is None:
            continue
        
        df = globals()[varname]
        
        # Filter by climate and component
        pred_data = df[(df['component'] == 'prediction') & (df['scenario'] == climate)]
        if len(pred_data) == 0:
            continue
        
        # Collect all latitude, truth, and prediction values for this climate
        all_lats = []
        all_truth = []
        all_pred = []
        
        for _, row in pred_data.iterrows():
            try:
                truth_vals = np.asarray(row['truth_values'], dtype=np.float64)
                pred_vals = np.asarray(row['pred_values'], dtype=np.float64)
                
                if truth_vals.size == 0 or pred_vals.size == 0:
                    continue
                
                # Reshape the 70 points as (10 lat, 7 lon)
                if truth_vals.size != n_lat_per_patch * n_lon_per_patch:
                    continue
                
                truth_2d = truth_vals.reshape(n_lat_per_patch, n_lon_per_patch)
                pred_2d = pred_vals.reshape(n_lat_per_patch, n_lon_per_patch)
                
                # For each latitude bin
                for lat_bin in range(n_lat_per_patch):
                    lat_val = lat_min + (lat_bin + 0.5) * (lat_max - lat_min) / n_lat_per_patch
                    
                    t_vals = truth_2d[lat_bin, :]
                    p_vals = pred_2d[lat_bin, :]
                    
                    valid_mask = np.isfinite(t_vals) & np.isfinite(p_vals)
                    if np.any(valid_mask):
                        n_valid = np.sum(valid_mask)
                        all_lats.extend([lat_val] * n_valid)
                        all_truth.extend(t_vals[valid_mask])
                        all_pred.extend(p_vals[valid_mask])
            except Exception:
                continue
        
        if len(all_lats) == 0:
            continue
        
        all_lats = np.array(all_lats)
        all_truth = np.array(all_truth)
        all_pred = np.array(all_pred)
        
        # Bin by latitude
        truth_binned = []
        pred_binned = []
        
        for i in range(len(lat_bins_hires) - 1):
            mask = (all_lats >= lat_bins_hires[i]) & (all_lats < lat_bins_hires[i + 1])
            if np.any(mask):
                truth_binned.append(np.nanmean(all_truth[mask]))
                pred_binned.append(np.nanmean(all_pred[mask]))
            else:
                truth_binned.append(np.nan)
                pred_binned.append(np.nan)
        
        truth_binned = np.array(truth_binned)
        pred_binned = np.array(pred_binned)
        
        # Remove NaN values
        valid_mask = ~(np.isnan(truth_binned) | np.isnan(pred_binned))
        lat_valid = lat_centers_hires[valid_mask]
        truth_valid = truth_binned[valid_mask]
        pred_valid = pred_binned[valid_mask]
        
        if truth_valid_all is None:
            truth_valid_all = truth_valid
            lat_valid_all = lat_valid
        
        if len(lat_valid) > 2:
            # Smooth
            sigma = 1.0
            pred_smooth = gaussian_filter1d(pred_valid, sigma=sigma, mode='nearest')
            
            # Plot prediction
            ax.plot(lat_valid, pred_smooth, linestyle=linestyle, linewidth=1.8, 
                    label=f'{label}', color=color, marker='', markersize=5, alpha=0.85)
    
    # Plot truth (shared reference) for this climate
    if truth_valid_all is not None and len(truth_valid_all) > 2:
        truth_smooth = gaussian_filter1d(truth_valid_all, sigma=1.0, mode='nearest')
        ax.plot(lat_valid_all, truth_smooth, linestyle='--', linewidth=1.8, 
                label='Ground Truth', color='black', marker='', markersize=5, alpha=0.85)
    
    ax.set_xlabel('Latitude (°)', fontsize=11, fontweight='bold')
    ax.set_ylabel(_ylabel, fontsize=11, fontweight='bold')
    ax.set_title(f'{climate.upper()}', fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3, linestyle=':')
    ax.legend(fontsize=9, loc='best', framealpha=0.95)

plt.suptitle(f'{variable} Profile by Latitude: Observed vs Predicted (by Climate Scenario)', 
             fontsize=15, fontweight='bold', y=1.02)
plt.show()

Confusion matrix

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────────────
VIZ_SETUP_CONF   = "CERA"   
VIZ_CLIMATE_CONF = "ssp585"          # "historical", "ssp126", "ssp245", "ssp370", "ssp585"
N_BINS_CONF      = 20        # number of quantile bins for the confusion matrix
NORMALIZE_CONF   = True      # True → normalised per row, False → raw counts
# ─────────────────────────────────────────────────────────────────────────────────────

_conf_sources = prediction_plot_sources_dict


if VIZ_SETUP_CONF not in _conf_sources:
    raise ValueError(f"Setup '{VIZ_SETUP_CONF}' unknown. Available: {list(_conf_sources.keys())}")

_df_conf = _conf_sources[VIZ_SETUP_CONF]
_pred_conf = _df_conf[
    (_df_conf["component"] == "prediction") &
    (_df_conf["scenario"] == VIZ_CLIMATE_CONF)
]

if _pred_conf.empty:
    raise ValueError(
        f"No prediction data for setup='{VIZ_SETUP_CONF}', climate='{VIZ_CLIMATE_CONF}'."
    )

# ── Collection of truth and prediction values ───────────────────────────────────────
_all_truth_c = []
_all_pred_c  = []

for _, _row in _pred_conf.iterrows():
    _t = np.asarray(_row["truth_values"], dtype=float).ravel()
    _p = np.asarray(_row["pred_values"],  dtype=float).ravel()
    _m = np.isfinite(_t) & np.isfinite(_p)
    _all_truth_c.extend(_t[_m].tolist())
    _all_pred_c.extend(_p[_m].tolist())

_all_truth_c = np.array(_all_truth_c)
_all_pred_c  = np.array(_all_pred_c)

if len(_all_truth_c) == 0:
    raise ValueError("No finite value available for the confusion matrix.")

# ── Bins per percentile (calculated on truth values) ───────────────────────────────
_bin_edges = np.unique(np.percentile(_all_truth_c, np.linspace(0, 100, N_BINS_CONF + 1)))
_n_bins    = len(_bin_edges) - 1

if _n_bins < 2:
    raise ValueError(
        "Not enough distinct values to create quantile bins. "
        "Try reducing N_BINS_CONF."
    )

# Truth: inner bounds → indices 0..n_bins-1
_truth_idx = np.clip(np.digitize(_all_truth_c, _bin_edges[1:-1]), 0, _n_bins - 1)

# Prediction: full bounds → indices 0..n_bins+1
#   0         = value < min(truth)   → "< min" column
#   1..n_bins = normal bin            → bin columns
#   n_bins+1  = value ≥ max(truth)   → "≥ max" column
_pred_idx_ext = np.digitize(_all_pred_c, _bin_edges)
_n_cols       = _n_bins + 2

# ── Filling the matrix (n_bins rows × n_cols columns) ───────────────────────
_conf_mat = np.zeros((_n_bins, _n_cols), dtype=float)
np.add.at(_conf_mat, (_truth_idx, _pred_idx_ext), 1)

if NORMALIZE_CONF:
    _row_sums = _conf_mat.sum(axis=1, keepdims=True)
    _row_sums[_row_sums == 0] = 1
    _conf_plot  = _conf_mat / _row_sums
    _cbar_label = "Fraction (normalized per row)"
    _annot_fmt  = ".2f"
else:
    _conf_plot  = _conf_mat
    _cbar_label = "Number of points"
    _annot_fmt  = ".0f"

# ── Labels ────────────────────────────────────────────────────────────────────────────
_bin_labels = [
    f"[{_bin_edges[i]:.2g}, {_bin_edges[i+1]:.2g})"
    for i in range(_n_bins)
]
_col_labels = (
    [f"< {_bin_edges[0]:.2g}"]
    + _bin_labels
    + [f"≥ {_bin_edges[-1]:.2g}"]
)

# ── Plot ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(
    figsize=(max(8, 7), 7),
    constrained_layout=True,
)

_im = ax.imshow(_conf_plot, aspect="auto", cmap="Blues", origin="lower",
                vmin=0, vmax=_conf_plot.max())

# Text annotations
for _i in range(_n_bins):
    for _j in range(_n_cols):
        _val        = _conf_plot[_i, _j]
        _text_color = "white" if _val > 0.55 * _conf_plot.max() else "black"
        ax.text(_j, _i, f"{_val:{_annot_fmt}}",
                ha="center", va="center", fontsize=7, color=_text_color)

# Diagonal highlighted (columns 1..n_bins, shifted by +1 because of "< min")
for _k in range(_n_bins):
    ax.add_patch(plt.Rectangle((_k + 0.5, _k - 0.5), 1, 1,
                                fill=False, edgecolor="#E63946", linewidth=1.2))

# Visual separators for the overflow columns
ax.axvline(0.5,              color="#6B7280", linewidth=1.5, linestyle="--", alpha=0.6)
ax.axvline(_n_cols - 1 - 0.5, color="#6B7280", linewidth=1.5, linestyle="--", alpha=0.6)

ax.set_xticks(range(_n_cols))
ax.set_xticklabels(_col_labels, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(_n_bins))
ax.set_yticklabels(_bin_labels, fontsize=8)
ax.set_xlabel("Predicted bin", fontweight="bold")
ax.set_ylabel("True bin (quantile)", fontweight="bold")
ax.set_title(
    f"Confusion matrix  ·  {VIZ_SETUP_CONF}  ·  {VIZ_CLIMATE_CONF}  ·  {variable}\n"
    f"({'normalized per row' if NORMALIZE_CONF else 'raw count'}"
    f", {len(_all_truth_c):,} points, {_n_bins} quantile bins)",
    fontweight="bold",
)

fig.colorbar(_im, ax=ax, label=_cbar_label, fraction=0.046, pad=0.04)
plt.show()

print(f"min truth : {_all_truth_c.min():.5f}  |  max truth : {_all_truth_c.max():.5f}")
print(f"min pred  : {_all_pred_c.min():.5f}  |  max pred  : {_all_pred_c.max():.5f}")
print(f"pred < min truth : {(_all_pred_c < _all_truth_c.min()).sum()}")
print(f"pred > max truth : {(_all_pred_c > _all_truth_c.max()).sum()}")

Confusion matrix on extreme precipitations

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────────────
VIZ_CLIMATE_CONF     = "ssp585"  # "historical", "ssp126", "ssp245", "ssp370", "ssp585"
N_BINS_CONF          = 20        # number of quantile bins
NORMALIZE_CONF       = True      # True → normalize per row (fraction of the true bin)
PRECIP_THRESHOLD_CONF = 10.0     # mm/day — only keep points where truth >= threshold
# ─────────────────────────────────────────────────────────────────────────────────────

_conf_sources = prediction_plot_sources_dict


def _compute_confusion_matrix(df, climate, n_bins, normalize, threshold=None):
    """Build a quantile-binned confusion matrix (truth vs. prediction) for one setup."""
    pred_df = df[(df["component"] == "prediction") & (df["scenario"] == climate)]
    if pred_df.empty:
        raise ValueError(f"No prediction data for climate='{climate}'.")

    # ── Collect all (truth, pred) pairs ────────────────────────────────────────────
    all_truth = []
    all_pred = []
    for _, row in pred_df.iterrows():
        t = np.asarray(row["truth_values"], dtype=float).ravel()
        p = np.asarray(row["pred_values"], dtype=float).ravel()
        m = np.isfinite(t) & np.isfinite(p)
        all_truth.extend(t[m].tolist())
        all_pred.extend(p[m].tolist())

    all_truth = np.array(all_truth)
    all_pred = np.array(all_pred)

    # ── Filter: keep only points where truth >= threshold ──────────────────────────
    if threshold is not None:
        keep = all_truth >= threshold
        all_truth = all_truth[keep]
        all_pred = all_pred[keep]

    if len(all_truth) == 0:
        raise ValueError("No finite values available for the confusion matrix.")

    # ── Quantile bins (computed on truth values) ───────────────────────────────────
    bin_edges = np.unique(np.percentile(all_truth, np.linspace(0, 100, n_bins + 1)))
    n_bins_eff = len(bin_edges) - 1
    if n_bins_eff < 2:
        raise ValueError(
            "Not enough distinct values to build quantile bins. Try lowering N_BINS_CONF."
        )

    # Truth: inner edges → indices 0..n_bins_eff-1
    truth_idx = np.clip(np.digitize(all_truth, bin_edges[1:-1]), 0, n_bins_eff - 1)

    # Prediction: full edges → indices 0..n_bins_eff+1
    #   0             = value < min(truth)   → "< min" column
    #   1..n_bins_eff  = normal bin           → bin columns
    #   n_bins_eff+1   = value ≥ max(truth)   → "≥ max" column
    pred_idx_ext = np.digitize(all_pred, bin_edges)
    n_cols = n_bins_eff + 2

    # ── Fill the matrix (n_bins_eff rows × n_cols columns) ─────────────────────────
    conf_mat = np.zeros((n_bins_eff, n_cols), dtype=float)
    np.add.at(conf_mat, (truth_idx, pred_idx_ext), 1)

    if normalize:
        row_sums = conf_mat.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        conf_plot = conf_mat / row_sums
    else:
        conf_plot = conf_mat

    bin_labels = [f"[{bin_edges[i]:.2g}, {bin_edges[i+1]:.2g})" for i in range(n_bins_eff)]
    col_labels = [f"< {bin_edges[0]:.2g}"] + bin_labels + [f"\u2265 {bin_edges[-1]:.2g}"]

    return {
        "conf_plot": conf_plot,
        "n_bins": n_bins_eff,
        "n_cols": n_cols,
        "bin_labels": bin_labels,
        "col_labels": col_labels,
        "all_truth": all_truth,
        "all_pred": all_pred,
    }


_results = {
    name: _compute_confusion_matrix(df, VIZ_CLIMATE_CONF, N_BINS_CONF, NORMALIZE_CONF,
                                     threshold=PRECIP_THRESHOLD_CONF)
    for name, df in _conf_sources.items()
}

_cbar_label = "Fraction (row-normalized)" if NORMALIZE_CONF else "Point count"
_annot_fmt = ".2f" if NORMALIZE_CONF else ".0f"
_global_vmax = max(res["conf_plot"].max() for res in _results.values())

# ── Plot: one matrix per setup, side by side ───────────────────────────────────────────
_n_setups = len(_results)
fig, axes = plt.subplots(1, _n_setups, figsize=(7 * _n_setups, 7), constrained_layout=True)
if _n_setups == 1:
    axes = [axes]

for ax, (setup_name, res) in zip(axes, _results.items()):
    conf_plot = res["conf_plot"]
    n_bins = res["n_bins"]
    n_cols = res["n_cols"]

    im = ax.imshow(conf_plot, aspect="auto", cmap="Blues", origin="lower",
                    vmin=0, vmax=_global_vmax)

    # Text annotations
    for i in range(n_bins):
        for j in range(n_cols):
            val = conf_plot[i, j]
            text_color = "white" if val > 0.55 * _global_vmax else "black"
            ax.text(j, i, f"{val:{_annot_fmt}}",
                    ha="center", va="center", fontsize=7, color=text_color)

    # Highlight the diagonal (columns 1..n_bins, shifted by +1 because of "< min")
    for k in range(n_bins):
        ax.add_patch(plt.Rectangle((k + 0.5, k - 0.5), 1, 1,
                                    fill=False, edgecolor="#E63946", linewidth=1.2))

    # Visual separators for the overflow columns
    ax.axvline(0.5,              color="#6B7280", linewidth=1.5, linestyle="--", alpha=0.6)
    ax.axvline(n_cols - 1 - 0.5, color="#6B7280", linewidth=1.5, linestyle="--", alpha=0.6)

    ax.set_xticks(range(n_cols))
    ax.set_xticklabels(res["col_labels"], rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(n_bins))
    ax.set_yticklabels(res["bin_labels"], fontsize=8)
    ax.set_xlabel("Predicted bin", fontweight="bold")
    ax.set_ylabel("True bin (quantile)", fontweight="bold")
    ax.set_title(
        f"{setup_name}  \u00b7  {VIZ_CLIMATE_CONF}  \u00b7  {variable}\n"
        f"(truth \u2265 {PRECIP_THRESHOLD_CONF:.0f} mm/day, "
        f"{'row-normalized' if NORMALIZE_CONF else 'raw count'}"
        f", {len(res['all_truth']):,} points, {n_bins} quantile bins)",
        fontweight="bold", fontsize=10,
    )

fig.colorbar(im, ax=axes, label=_cbar_label, fraction=0.02, pad=0.02)
plt.show()

for setup_name, res in _results.items():
    all_truth, all_pred = res["all_truth"], res["all_pred"]
    print(f"[{setup_name}]")
    print(f"  min truth : {all_truth.min():.5f}  |  max truth : {all_truth.max():.5f}")
    print(f"  min pred  : {all_pred.min():.5f}  |  max pred  : {all_pred.max():.5f}")
    print(f"  pred < min truth : {(all_pred < all_truth.min()).sum()}")
    print(f"  pred > max truth : {(all_pred > all_truth.max()).sum()}")


Vizualisation of prediction over a sample

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────────────
VIZ_SETUP_PRED   = "CERA"   # main setup: "CERA", "Baseline no align", "Baseline simple"
VIZ_CLIMATE_PRED = "ssp585"
VIZ_SAMPLE_IDX   = 19       # sample index within the set (main setup, climate)
# ─────────────────────────────────────────────────────────────────────────────────────

# ── Patch catalog ─────────────────────────────────────────────────────────────────────
_catalog_path = None
for _name in ("patch_catalog.pkl", "patches_catalog.pkl", "patch_catalog.csv"):
    _p = precomputed_dir / _name
    if _p.exists():
        _catalog_path = _p
        break
if _catalog_path is None:
    raise FileNotFoundError(f"patch_catalog not found in {precomputed_dir}.")

if _catalog_path.suffix == ".pkl":
    with open(_catalog_path, "rb") as _f:
        _patch_catalog = pickle.load(_f)
else:
    _patch_catalog = pd.read_csv(_catalog_path)


def _get_patch_lat_lon_grids(patch_id):
    rows = _patch_catalog.loc[_patch_catalog["patch_id"] == int(patch_id)]
    if rows.empty:
        raise ValueError(f"patch_id={patch_id} not found in patch_catalog.")
    info = rows.iloc[0]
    lats = np.linspace(float(info["lat_start"]), float(info["lat_stop"]),
                       int(info["lat_stop_idx"] - info["lat_start_idx"]))
    lons = np.linspace(float(info["lon_start"]), float(info["lon_stop"]),
                       int(info["lon_stop_idx"] - info["lon_start_idx"]))
    lon_grid, lat_grid = np.meshgrid(lons, lats)
    return lat_grid, lon_grid, info


_pred_setup_dict = _conf_sources


# ── Main sample selection ─────────────────────────────────────────────────────────────
if VIZ_SETUP_PRED not in _pred_setup_dict:
    raise ValueError(f"Setup '{VIZ_SETUP_PRED}' unknown.")

_src_df = _pred_setup_dict[VIZ_SETUP_PRED]
_main_df = _src_df[
    (_src_df["component"] == "prediction") &
    (_src_df["scenario"] == VIZ_CLIMATE_PRED)
].reset_index(drop=True)

if _main_df.empty:
    raise ValueError(f"No data for setup='{VIZ_SETUP_PRED}', climate='{VIZ_CLIMATE_PRED}'.")
if VIZ_SAMPLE_IDX >= len(_main_df):
    raise IndexError(f"VIZ_SAMPLE_IDX={VIZ_SAMPLE_IDX} out of bounds (max {len(_main_df)-1}).")

_row_main = _main_df.iloc[VIZ_SAMPLE_IDX]
_patch_id = int(_row_main["patch_id"])

# ── Geographic grid (shared across all panels) ─────────────────────────────────────────
_lat_grid, _lon_grid, _patch_info = _get_patch_lat_lon_grids(_patch_id)
_n_lat_pp, _n_lon_pp = _lat_grid.shape
_lon_min, _lon_max = float(_lon_grid.min()), float(_lon_grid.max())
_lat_min, _lat_max = float(_lat_grid.min()), float(_lat_grid.max())
_margin_lon = max(1.0, 0.2 * (_lon_max - _lon_min))
_margin_lat = max(1.0, 0.2 * (_lat_max - _lat_min))
_extent_patch = [_lon_min - _margin_lon, _lon_max + _margin_lon,
                 _lat_min - _margin_lat, _lat_max + _margin_lat]


def _to_2d(flat):
    return np.asarray(flat, dtype=float).reshape(_n_lat_pp, _n_lon_pp)


# ── Ground truth extraction ─────────────────────────────────────────────────────────────
_truth_2d = _to_2d(_row_main["truth_values"])
_panels   = []

# ── Matching all setups by patch_id ────────────────────────────────────────────────────
for _sname, _sdf in _pred_setup_dict.items():
    _sdf_filt = _sdf[
        (_sdf["component"] == "prediction") &
        (_sdf["scenario"] == VIZ_CLIMATE_PRED) &
        (_sdf["patch_id"] == _patch_id)
    ].reset_index(drop=True)

    if _sdf_filt.empty:
        print(f"[WARN] No sample found for {_sname} / patch_id={_patch_id} / {VIZ_CLIMATE_PRED} — panel omitted.")
        _panels.append((_sname, None))
    else:
        _panels.append((_sname, _to_2d(_sdf_filt.iloc[0]["pred_values"])))


# ── Shared color limits (truth + all available predictions) ──────────────────────────
_all_arrays = [_truth_2d] + [arr for _, arr in _panels if arr is not None]
_vmin = float(min(np.nanmin(a) for a in _all_arrays))
_vmax = float(max(np.nanmax(a) for a in _all_arrays))
_cmap = "Blues"
_proj = ccrs.PlateCarree()

# ── Figure: N predictions | Ground Truth | World map ──────────────────────────────────
_n_pred = len(_panels)   # number of prediction panels (main + baselines)
import math
_n_cols = math.ceil((_n_pred + 1) / 2)  # +1 for the ground truth


fig = plt.figure(figsize=(4.5 * _n_cols + 2.5, 11), constrained_layout=True)
gs  = gridspec.GridSpec(2, _n_cols + 1, figure=fig)


def _patch_panel(pos):
    ax = fig.add_subplot(pos, projection=_proj)
    ax.set_extent(_extent_patch, crs=_proj)
    ax.coastlines(resolution="110m", linewidth=0.8, color="black")
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, edgecolor="gray")
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.35)
    gl.top_labels   = False
    gl.right_labels = False
    return ax


# ── Prediction panels ─────────────────────────────────────────────────────────────────
_im = None
_colored_axes = []

for col, (label, pred_2d) in enumerate(_panels):
    ax = _patch_panel(gs[col // _n_cols, col % _n_cols])
    _colored_axes.append(ax)
    if pred_2d is not None:
        _im = ax.pcolormesh(
            _lon_grid, _lat_grid, pred_2d,
            shading="auto", cmap=_cmap, vmin=_vmin, vmax=_vmax, transform=_proj,
        )
        ax.scatter(_lon_grid.ravel(), _lat_grid.ravel(),
                   s=8, color="black", alpha=0.4, transform=_proj, zorder=4)
    else:
        ax.text(0.5, 0.5, "N/A", transform=ax.transAxes,
                ha="center", va="center", fontsize=14, color="gray")
    ax.set_title(f"Prediction  ({label})\n{variable}  ·  {VIZ_CLIMATE_PRED}  ·  patch {_patch_id}",
                 fontweight="bold")

# ── Ground Truth panel ────────────────────────────────────────────────────────────────
ax_truth = _patch_panel(gs[_n_pred // _n_cols, _n_pred % _n_cols])

_colored_axes.append(ax_truth)
_im_truth = ax_truth.pcolormesh(
    _lon_grid, _lat_grid, _truth_2d,
    shading="auto", cmap=_cmap, vmin=_vmin, vmax=_vmax, transform=_proj,
)
ax_truth.scatter(_lon_grid.ravel(), _lat_grid.ravel(),
                 s=8, color="black", alpha=0.4, transform=_proj, zorder=4)
ax_truth.set_title(f"Ground Truth\n{variable}  ·  {VIZ_CLIMATE_PRED}  ·  sample #{VIZ_SAMPLE_IDX}",
                   fontweight="bold")

# ── Shared colorbar ───────────────────────────────────────────────────────────────────
if _im is not None:
    fig.colorbar(_im, ax=_colored_axes, orientation="vertical",
                 fraction=0.02, pad=0.02, label=variable)

# ── World map ─────────────────────────────────────────────────────────────────────────
ax_world = fig.add_subplot(gs[1, _n_cols], projection=_proj)
ax_world.set_global()
ax_world.coastlines(resolution="110m", linewidth=0.7, color="black")
ax_world.add_feature(cfeature.BORDERS, linewidth=0.3, edgecolor="gray")
ax_world.gridlines(linewidth=0.3, alpha=0.35)
ax_world.add_patch(mpatches.Rectangle(
    (_lon_min, _lat_min), _lon_max - _lon_min, _lat_max - _lat_min,
    linewidth=2.0, edgecolor="red", facecolor="none", transform=_proj, zorder=5,
))
ax_world.set_title(f"Location\npatch_id = {_patch_id}", fontweight="bold")

fig.suptitle(
    f"Prediction vs Ground Truth  ·  {VIZ_CLIMATE_PRED}  ·  {variable}",
    fontweight="bold", fontsize=13,
)
plt.show()

Log densities

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────────────
VIZ_CLIMATE_DIST = "ssp585"
VIZ_N_LOG_BINS   = 40
PR_DRY_THRESHOLD = 0.1      # values below are ignored for the log axis
# ─────────────────────────────────────────────────────────────────────────────────────

_pred_setup_dict = prediction_plot_sources_dict


_n = len(_pred_setup_dict)
fig, axes = plt.subplots(2, _n, figsize=(6 * _n, 10))

_valid_cols = []

for col, (setup_name, src_df) in enumerate(_pred_setup_dict.items()):
    ax_hist = axes[0, col]
    ax_surv = axes[1, col]

    _mask = src_df["component"] == "prediction"
    if VIZ_CLIMATE_DIST is not None:
        _mask &= src_df["scenario"] == VIZ_CLIMATE_DIST
    _df = src_df[_mask]

    if _df.empty:
        ax_hist.set_title(f"{setup_name}\n(no data)")
        continue

    _valid_cols.append(col)

    _truth_all = np.concatenate([np.asarray(v, dtype=float).ravel() for v in _df["truth_values"]])
    _pred_all  = np.concatenate([np.asarray(v, dtype=float).ravel() for v in _df["pred_values"]])

    # Filtering positive values for the log axis
    _truth_wet = _truth_all[_truth_all >= PR_DRY_THRESHOLD]
    _pred_wet  = _pred_all[_pred_all   >= PR_DRY_THRESHOLD]

    _log_min = np.log10(PR_DRY_THRESHOLD)
    _log_max = np.log10(max(_truth_wet.max(), _pred_wet.max()) * 1.1)
    _bins = np.logspace(_log_min, _log_max, VIZ_N_LOG_BINS + 1)

    # ── Row 1: log-spaced histogram ───────────────────────────────────────────────────
    ax_hist.hist(_truth_wet, bins=_bins, density=True, alpha=0.4,
                 color="steelblue", label=f"Ground Truth (wet: {100*len(_truth_wet)/len(_truth_all):.1f}%)")
    ax_hist.hist(_pred_wet,  bins=_bins, density=True, alpha=0.4,
                 color="tomato",    label=f"Prediction  (wet: {100*len(_pred_wet)/len(_pred_all):.1f}%)")
    ax_hist.set_xscale("log")

    # KDE in log space → transformed back to linear density via the Jacobian
    for _vals, _col in [(_truth_wet, "steelblue"), (_pred_wet, "tomato")]:
        _log_vals = np.log10(_vals)
        _kde = gaussian_kde(_log_vals, bw_method="silverman")
        _xs_log = np.linspace(_log_vals.min(), _log_vals.max(), 400)
        _xs_lin = 10 ** _xs_log
        _dens_lin = _kde(_xs_log) / (_xs_lin * np.log(10))  # Jacobian of the change of variable
        ax_hist.plot(_xs_lin, _dens_lin, color=_col, linewidth=2)

    # Reference lines
    _p99_t = np.percentile(_truth_all, 99)
    _p99_p = np.percentile(_pred_all,  99)
    ax_hist.axvline(20.0,   color="black",    linestyle=":",  linewidth=1.2, alpha=0.7, label="20 mm/day")
    ax_hist.axvline(_p99_t, color="steelblue", linestyle="--", linewidth=1.2, alpha=0.8, label=f"Truth p99={_p99_t:.1f}")
    ax_hist.axvline(_p99_p, color="tomato",    linestyle="--", linewidth=1.2, alpha=0.8, label=f"Pred  p99={_p99_p:.1f}")

    _txt = (
        f"Truth: p95={np.percentile(_truth_all,95):.3g}  p99={_p99_t:.3g}  p99.9={np.percentile(_truth_all,99.9):.3g}\n"
        f"Pred:  p95={np.percentile(_pred_all, 95):.3g}  p99={_p99_p:.3g}  p99.9={np.percentile(_pred_all, 99.9):.3g}"
    )
    ax_hist.text(0.98, 0.98, _txt, transform=ax_hist.transAxes,
                 ha="right", va="top", fontsize=8, family="monospace",
                 bbox=dict(boxstyle="round,pad=0.35", facecolor="white", alpha=0.8))

    _clim = VIZ_CLIMATE_DIST or "all"
    ax_hist.set_title(f"{setup_name}  ·  {_clim}  ·  pr (log)", fontweight="bold")
    ax_hist.set_xlabel("pr (mm/day)")
    ax_hist.set_ylabel("Density (values ≥ 0.1 mm/day)")
    ax_hist.legend(fontsize=8)
    ax_hist.grid(True, alpha=0.3, which="both")

    # ── Row 2: survival function P(X > x) in log-log ─────────────────────────────────
    for _vals, _col, _lbl in [
        (_truth_all, "steelblue", "Ground Truth"),
        (_pred_all,  "tomato",    "Prediction"),
    ]:
        _sorted = np.sort(_vals)
        _surv = 1.0 - np.arange(1, len(_sorted) + 1) / len(_sorted)
        _pos = (_sorted > 0) & (_surv > 0)
        ax_surv.plot(_sorted[_pos], _surv[_pos], color=_col, linewidth=2, label=_lbl)

    ax_surv.set_xscale("log")
    ax_surv.set_yscale("log")
    ax_surv.axvline(20.0,  color="black", linestyle=":",  linewidth=1.2, alpha=0.7, label="20 mm/day")
    ax_surv.axhline(0.01,  color="gray",  linestyle="--", linewidth=1.0, alpha=0.8, label="p99  (1%)")
    ax_surv.axhline(0.001, color="gray",  linestyle=":",  linewidth=1.0, alpha=0.6, label="p99.9 (0.1%)")
    ax_surv.set_title(f"P(X > x)  ·  {setup_name}", fontweight="bold")
    ax_surv.set_xlabel("pr (mm/day)")
    ax_surv.set_ylabel("P(X > x)")
    ax_surv.legend(fontsize=8)
    ax_surv.grid(True, alpha=0.3, which="both")

# ── Harmonizing x/y scales for each plot type (comparability) ──────────────────────
if _valid_cols:
    _hist_axes = [axes[0, c] for c in _valid_cols]
    _hist_xmin = min(ax.get_xlim()[0] for ax in _hist_axes)
    _hist_xmax = max(ax.get_xlim()[1] for ax in _hist_axes)
    _hist_ymin = min(ax.get_ylim()[0] for ax in _hist_axes)
    _hist_ymax = max(ax.get_ylim()[1] for ax in _hist_axes)
    for ax in _hist_axes:
        ax.set_xlim(_hist_xmin, _hist_xmax)
        ax.set_ylim(_hist_ymin, _hist_ymax)

    _surv_axes = [axes[1, c] for c in _valid_cols]
    _surv_xmin = min(ax.get_xlim()[0] for ax in _surv_axes)
    _surv_xmax = max(ax.get_xlim()[1] for ax in _surv_axes)
    _surv_ymin = min(ax.get_ylim()[0] for ax in _surv_axes)
    _surv_ymax = max(ax.get_ylim()[1] for ax in _surv_axes)
    for ax in _surv_axes:
        ax.set_xlim(_surv_xmin, _surv_xmax)
        ax.set_ylim(_surv_ymin, _surv_ymax)

fig.suptitle(
    f"pr log-scale distribution  ·  Ground Truth vs Prediction",
    fontweight="bold", fontsize=14,
)
plt.tight_layout() 
plt.show()


Log densities on extreme precipitations

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────────────
VIZ_CLIMATE_DIST      = "ssp245"
VIZ_N_LOG_BINS        = 40
PR_DRY_THRESHOLD      = 0.1    # log-axis threshold (not used for the main filter)
HEAVY_PRECIP_THRESHOLD = 10.0  # only values above this threshold are kept (mm/day)
# ─────────────────────────────────────────────────────────────────────────────────────

_pred_setup_dict = prediction_plot_sources_dict


_n = len(_pred_setup_dict)
fig, axes = plt.subplots(2, _n, figsize=(6 * _n, 10))

for col, (setup_name, src_df) in enumerate(_pred_setup_dict.items()):
    ax_hist = axes[0, col]
    ax_surv = axes[1, col]

    _mask = src_df["component"] == "prediction"
    if VIZ_CLIMATE_DIST is not None:
        _mask &= src_df["scenario"] == VIZ_CLIMATE_DIST
    _df = src_df[_mask]

    if _df.empty:
        ax_hist.set_title(f"{setup_name}\n(no data)")
        continue

    _truth_raw = np.concatenate([np.asarray(v, dtype=float).ravel() for v in _df["truth_values"]])
    _pred_raw  = np.concatenate([np.asarray(v, dtype=float).ravel() for v in _df["pred_values"]])

    # ── Main filter: precipitation > HEAVY_PRECIP_THRESHOLD mm/day ───────────────────
    _truth_all = _truth_raw[_truth_raw > HEAVY_PRECIP_THRESHOLD]
    _pred_all  = _pred_raw[_pred_raw   > HEAVY_PRECIP_THRESHOLD]

    if len(_truth_all) == 0 or len(_pred_all) == 0:
        ax_hist.set_title(f"{setup_name}\n(no value > {HEAVY_PRECIP_THRESHOLD} mm/day)")
        continue

    # For the log-spaced histogram: all values are already above the threshold
    _truth_wet = _truth_all
    _pred_wet  = _pred_all

    _log_min = np.log10(HEAVY_PRECIP_THRESHOLD)
    _log_max = np.log10(max(_truth_wet.max(), _pred_wet.max()) * 1.1)
    _bins = np.logspace(_log_min, _log_max, VIZ_N_LOG_BINS + 1)

    # ── Row 1: log-spaced histogram ───────────────────────────────────────────────────
    ax_hist.hist(_truth_wet, bins=_bins, density=True, alpha=0.4,
                 color="steelblue", label=f"Ground Truth ({len(_truth_wet):,} pts)")
    ax_hist.hist(_pred_wet,  bins=_bins, density=True, alpha=0.4,
                 color="tomato",    label=f"Prediction  ({len(_pred_wet):,} pts)")
    ax_hist.set_xscale("log")

    # KDE in log space → transformed back to linear density via the Jacobian
    for _vals, _col in [(_truth_wet, "steelblue"), (_pred_wet, "tomato")]:
        _log_vals = np.log10(_vals)
        _kde = gaussian_kde(_log_vals, bw_method="silverman")
        _xs_log = np.linspace(_log_vals.min(), _log_vals.max(), 400)
        _xs_lin = 10 ** _xs_log
        _dens_lin = _kde(_xs_log) / (_xs_lin * np.log(10))
        ax_hist.plot(_xs_lin, _dens_lin, color=_col, linewidth=2)

    # Reference lines
    _p99_t = np.percentile(_truth_all, 99)
    _p99_p = np.percentile(_pred_all,  99)
    ax_hist.axvline(20.0,    color="black",     linestyle=":",  linewidth=1.2, alpha=0.7, label="20 mm/day")
    ax_hist.axvline(_p99_t,  color="steelblue", linestyle="--", linewidth=1.2, alpha=0.8, label=f"Truth p99={_p99_t:.1f}")
    ax_hist.axvline(_p99_p,  color="tomato",    linestyle="--", linewidth=1.2, alpha=0.8, label=f"Pred  p99={_p99_p:.1f}")

    _txt = (
        f"Truth: p95={np.percentile(_truth_all,95):.3g}  p99={_p99_t:.3g}  p99.9={np.percentile(_truth_all,99.9):.3g}\n"
        f"Pred:  p95={np.percentile(_pred_all, 95):.3g}  p99={_p99_p:.3g}  p99.9={np.percentile(_pred_all, 99.9):.3g}"
    )
    ax_hist.text(0.98, 0.98, _txt, transform=ax_hist.transAxes,
                 ha="right", va="top", fontsize=8, family="monospace",
                 bbox=dict(boxstyle="round,pad=0.35", facecolor="white", alpha=0.8))

    _clim = VIZ_CLIMATE_DIST or "all"
    ax_hist.set_title(f"{setup_name}  ·  {_clim}  ·  pr > {HEAVY_PRECIP_THRESHOLD} mm/day (log)", fontweight="bold")
    ax_hist.set_xlabel("pr (mm/day)")
    ax_hist.set_ylabel(f"Density (pr > {HEAVY_PRECIP_THRESHOLD} mm/day)")
    ax_hist.legend(fontsize=8)
    ax_hist.grid(True, alpha=0.3, which="both")

    # ── Row 2: survival function P(X > x) in log-log ─────────────────────────────────
    for _vals, _col, _lbl in [
        (_truth_all, "steelblue", "Ground Truth"),
        (_pred_all,  "tomato",    "Prediction"),
    ]:
        _sorted = np.sort(_vals)
        _surv = 1.0 - np.arange(1, len(_sorted) + 1) / len(_sorted)
        _pos = (_sorted > 0) & (_surv > 0)
        ax_surv.plot(_sorted[_pos], _surv[_pos], color=_col, linewidth=2, label=_lbl)

    ax_surv.set_xscale("log")
    ax_surv.set_yscale("log")
    ax_surv.axvline(20.0,  color="black", linestyle=":",  linewidth=1.2, alpha=0.7, label="20 mm/day")
    ax_surv.axhline(0.01,  color="gray",  linestyle="--", linewidth=1.0, alpha=0.8, label="p99  (1%)")
    ax_surv.axhline(0.001, color="gray",  linestyle=":",  linewidth=1.0, alpha=0.6, label="p99.9 (0.1%)")
    ax_surv.set_title(f"Survival P(X > x)  ·  {setup_name}  ·  pr > {HEAVY_PRECIP_THRESHOLD} mm/day", fontweight="bold")
    ax_surv.set_xlabel("pr (mm/day)")
    ax_surv.set_ylabel("P(X > x)")
    ax_surv.legend(fontsize=8)
    ax_surv.grid(True, alpha=0.3, which="both")

fig.suptitle(
    f"pr log-scale distribution > {HEAVY_PRECIP_THRESHOLD} mm/day  ·  Ground Truth vs Prediction",
    fontweight="bold", fontsize=14,
)
plt.tight_layout()
plt.show()


Confusion matrix on qualitatives levels of rain

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────────────
VIZ_CLIMATE_RAIN = "ssp585"   # None → all climates, otherwise one specific climate
# ─────────────────────────────────────────────────────────────────────────────────────

RAIN_BINS   = [0.1, 1.0, 5.0, 10.0, 20.0]  # thresholds for np.digitize
RAIN_LABELS = ["No rain", "Very light rain", "Light rain",
                "Moderate rain", "Heavy rain", "Very heavy rain"]
N_CLASSES   = len(RAIN_LABELS)

_rain_sources = prediction_plot_sources_dict
_rain_setup_order = setup_order
_n_cols = math.ceil(len(_rain_setup_order) / 2)


fig, axes = plt.subplots(
2, _n_cols,
figsize=(6 * _n_cols, 11),
constrained_layout=True,
)


_last_im = None
for col, setup_name in enumerate(_rain_setup_order):
    ax = axes[col // _n_cols, col % _n_cols]
    src_df = _rain_sources[setup_name]

    _mask = src_df["component"] == "prediction"
    if VIZ_CLIMATE_RAIN is not None:
        _mask &= src_df["scenario"] == VIZ_CLIMATE_RAIN
    _df = src_df[_mask]

    if _df.empty:
        ax.set_title(f"{setup_name}\n(no data)")
        continue

    _truth_all = np.concatenate([np.asarray(v, dtype=float).ravel() for v in _df["truth_values"]])
    _pred_all  = np.concatenate([np.asarray(v, dtype=float).ravel() for v in _df["pred_values"]])
    _valid = np.isfinite(_truth_all) & np.isfinite(_pred_all)
    _truth_all = _truth_all[_valid]
    _pred_all  = _pred_all[_valid]

    _truth_cat = np.digitize(_truth_all, RAIN_BINS)
    _pred_cat  = np.digitize(_pred_all,  RAIN_BINS)

    _cm = np.zeros((N_CLASSES, N_CLASSES), dtype=float)
    for _t, _p in zip(_truth_cat, _pred_cat):
        _cm[_t, _p] += 1.0

    _row_sums = _cm.sum(axis=1, keepdims=True)
    _row_sums[_row_sums == 0] = 1.0
    _cm_norm = _cm / _row_sums

    _last_im = ax.imshow(_cm_norm, vmin=0.0, vmax=1.0, cmap="Blues", aspect="auto")
    ax.set_xticks(range(N_CLASSES))
    ax.set_yticks(range(N_CLASSES))
    ax.set_xticklabels(RAIN_LABELS, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(RAIN_LABELS, fontsize=8)
    ax.set_xlabel("Predicted class", fontsize=10)
    if col % _n_cols == 0:
        ax.set_ylabel("True class", fontsize=10)
    _title_suffix = f" ({VIZ_CLIMATE_RAIN})" if VIZ_CLIMATE_RAIN else " (all climates)"
    ax.set_title(f"{setup_name}{_title_suffix}", fontsize=11)

    for r in range(N_CLASSES):
        for c in range(N_CLASSES):
            _val = _cm_norm[r, c]
            _txt_color = "white" if _val > 0.6 else "black"
            ax.text(c, r, f"{_val:.2f}", ha="center", va="center",
                    fontsize=7, color=_txt_color)

if _last_im is not None:
    plt.colorbar(_last_im, ax=axes.ravel().tolist(), label="Fraction (row-normalized)", shrink=0.8)

_fig_title = "Rain intensity confusion matrix — pr"
_fig_title += f" ({VIZ_CLIMATE_RAIN})" if VIZ_CLIMATE_RAIN else " (all climates)"
fig.suptitle(_fig_title, fontsize=13, fontweight="bold")
plt.show()

RMSE on extreme precipitation

In [ ]:
EXTREME_THRESHOLD = 10.0  # mm/day — Very heavy rain

_extreme_sources = prediction_plot_sources_list
_extreme_setup_order = setup_order
_extreme_markers = setup_markers
_extreme_colors  = setup_colors

_extreme_rows = []
for _setup_name, _src_df in _extreme_sources:
    _pred_df = _src_df[_src_df["component"] == "prediction"]
    for _climate in _pred_df["scenario"].unique():
        _climate_df = _pred_df[_pred_df["scenario"] == _climate]

        _truth_list, _pred_list = [], []
        for _, _row in _climate_df.iterrows():
            _t = np.asarray(_row["truth_values"], dtype=float).ravel()
            _p = np.asarray(_row["pred_values"],  dtype=float).ravel()
            _m = np.isfinite(_t) & np.isfinite(_p)
            _truth_list.extend(_t[_m].tolist())
            _pred_list.extend(_p[_m].tolist())

        if not _truth_list:
            continue

        _truth_arr = np.asarray(_truth_list)
        _pred_arr  = np.asarray(_pred_list)

        _ext_mask = _truth_arr >= EXTREME_THRESHOLD
        if not np.any(_ext_mask):
            continue

        _rmse_ext = float(np.sqrt(np.mean((_truth_arr[_ext_mask] - _pred_arr[_ext_mask]) ** 2)))
        _extreme_rows.append({
            "setup":       _setup_name,
            "scenario":    _climate,
            "rmse_extreme": _rmse_ext,
        })

_extreme_df = pd.DataFrame(_extreme_rows)
if _extreme_df.empty:
    print(f"No data above {EXTREME_THRESHOLD} mm/day.")
else:
    _scenario_order_e = list(dict.fromkeys(_extreme_df["scenario"].tolist()))

    fig, ax = plt.subplots(figsize=(4, 4.0), constrained_layout=True)
    for _setup_name in _extreme_setup_order:
        _sub = _extreme_df[_extreme_df["setup"] == _setup_name].copy()
        if _sub.empty:
            continue
        _sub["scenario"] = pd.Categorical(_sub["scenario"], categories=_scenario_order_e, ordered=True)
        _sub = _sub.sort_values("scenario")
        ax.plot(
            _sub["scenario"],
            _sub["rmse_extreme"],
            linestyle="None",
            marker=_extreme_markers[_setup_name],
            markersize=7,
            color=_extreme_colors[_setup_name],
            label=_setup_name,
        )

    ax.set_xlabel("Scenario")
    ax.set_ylabel("RMSE (mm/day)")
    ax.set_title(f"RMSE on extreme precipitation\n(ground truth \u2265 {EXTREME_THRESHOLD:.0f} mm/day)")
    ax.grid(axis="y", alpha=0.25)
    ax.set_ylim(bottom=0, top=60)

    _legend_handles_e = [
        Line2D([0], [0], color=_extreme_colors[s], marker=_extreme_markers[s],
                linewidth=0.0, label=s)
        for s in _extreme_setup_order
    ]
    ax.legend(handles=_legend_handles_e, title="Setup")
    plt.xticks(rotation=20, ha="right")
    plt.show()

## Part 4 - Plotting history results

### *CERA*

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6), constrained_layout=True)

# Modern palette
colors = {
    "recon": "#2C7FB8",   # clean blue
    "align": "#7F7F7F",   # neutral gray
    "pred":  "#6A3D9A",   # discreet violet (optional but useful)
    "total": "#D62728",   # red
}

loss_groups = {
    "recon": ["train_recon_loss", "val_recon_loss"],
    "align": ["train_align_loss", "val_align_loss"],
    "pred":  ["train_pred_loss", "val_pred_loss"],
    "total": ["train_total_loss", "val_total_loss"],
}

for group, cols in loss_groups.items():
    for col in cols:
        is_train = "train" in col
        
        ax.plot(
            cera_history_df["epoch"],
            cera_history_df[col],
            linestyle="--" if is_train else "-",   # train dashed
            linewidth=1,
            color=colors[group],
            alpha=0.9 if not is_train else 0.7,
            label=col.replace("_", " ")
        )

ax.set_yscale("log")
ax.set_title("CERA training dynamics (log scale)", fontsize=13)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss (log scale)")

ax.grid(True, which="both", axis="y", alpha=0.25)
ax.grid(True, which="major", axis="x", alpha=0.15)

ax.legend(title="Loss components", ncol=2, fontsize=9)

plt.show()

### *CERA noAlign*

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6), constrained_layout=True)

# Modern palette
colors = {
    "recon": "#2C7FB8",   # clean blue
    "pred":  "#6A3D9A",   # discreet violet (optional but useful)
    "total": "#D62728",   # red
}

loss_groups = {
    "recon": ["train_recon_loss", "val_recon_loss"],
    "pred":  ["train_pred_loss", "val_pred_loss"],
    "total": ["train_total_loss", "val_total_loss"],
}

for group, cols in loss_groups.items():
    for col in cols:
        is_train = "train" in col
        
        ax.plot(
            baseline_cera_noalign_history_df["epoch"],
            baseline_cera_noalign_history_df[col],
            linestyle="--" if is_train else "-",   # train dashed
            linewidth=1,
            color=colors[group],
            alpha=0.9 if not is_train else 0.7,
            label=col.replace("_", " ")
        )

ax.set_yscale("log")
ax.set_title("CERA no-align training dynamics (log scale)", fontsize=13)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss (log scale)")

ax.grid(True, which="both", axis="y", alpha=0.25)
ax.grid(True, which="major", axis="x", alpha=0.15)

ax.legend(title="Loss components", ncol=2, fontsize=9)

plt.show()

### *CERA full latent*

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6), constrained_layout=True)

colors = {
    "recon": "#2C7FB8",
    "align": "#7F7F7F",
    "pred":  "#6A3D9A",
    "total": "#D62728",
}

loss_groups = {
    "recon": ["train_recon_loss", "val_recon_loss"],
    "align": ["train_align_loss", "val_align_loss"],
    "pred":  ["train_pred_loss", "val_pred_loss"],
    "total": ["train_total_loss", "val_total_loss"],
}

for group, cols in loss_groups.items():
    for col in cols:
        if col not in cera_full_latent_history_df.columns:
            continue
        is_train = "train" in col
        ax.plot(
            cera_full_latent_history_df["epoch"],
            cera_full_latent_history_df[col],
            linestyle="--" if is_train else "-",
            linewidth=1,
            color=colors[group],
            alpha=0.9 if not is_train else 0.7,
            label=col.replace("_", " ")
        )

ax.set_yscale("log")
ax.set_title("CERA full latent training dynamics (log scale)", fontsize=13)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss (log scale)")
ax.grid(True, which="both", axis="y", alpha=0.25)
ax.grid(True, which="major", axis="x", alpha=0.15)
ax.legend(title="Loss components", ncol=2, fontsize=9)
plt.show()

### *Baseline simple*

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6), constrained_layout=True)

colors = {"train": "#2C7FB8", "val": "#D62728"}

ax.plot(baseline_simple_history_df["epoch"], baseline_simple_history_df["train_loss"],
        linestyle="--", linewidth=1, color=colors["train"], alpha=0.7, label="train loss")
ax.plot(baseline_simple_history_df["epoch"], baseline_simple_history_df["val_loss"],
        linestyle="-",  linewidth=1, color=colors["val"],   alpha=0.9, label="val loss")

ax.set_yscale("log")
ax.set_title("Baseline simple training dynamics (log scale)", fontsize=13)
ax.set_xlabel("Epoch")
ax.set_ylabel("Prediction loss (log scale)")
ax.grid(True, which="both", axis="y", alpha=0.25)
ax.grid(True, which="major", axis="x", alpha=0.15)
ax.legend(title="Loss", fontsize=9)
plt.show()

### *Baseline physical*

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6), constrained_layout=True)

colors = {"train": "#2C7FB8", "val": "#D62728"}

ax.plot(baseline_physical_history_df["epoch"], baseline_physical_history_df["train_loss"],
        linestyle="--", linewidth=1, color=colors["train"], alpha=0.7, label="train loss")
ax.plot(baseline_physical_history_df["epoch"], baseline_physical_history_df["val_loss"],
        linestyle="-",  linewidth=1, color=colors["val"],   alpha=0.9, label="val loss")

ax.set_yscale("log")
ax.set_title("Baseline physical training dynamics (log scale)", fontsize=13)
ax.set_xlabel("Epoch")
ax.set_ylabel("Prediction loss (log scale)")
ax.grid(True, which="both", axis="y", alpha=0.25)
ax.grid(True, which="major", axis="x", alpha=0.15)
ax.legend(title="Loss", fontsize=9)
plt.show()

### *Baseline ClimaX*

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6), constrained_layout=True)

colors = {"train": "#2C7FB8", "val": "#D62728"}

ax.plot(baseline_climax_history_df["epoch"], baseline_climax_history_df["train_loss"],
        linestyle="--", linewidth=1, color=colors["train"], alpha=0.7, label="train loss")
ax.plot(baseline_climax_history_df["epoch"], baseline_climax_history_df["val_loss"],
        linestyle="-",  linewidth=1, color=colors["val"],   alpha=0.9, label="val loss")

ax.set_yscale("log")
ax.set_title("Baseline ClimaX training dynamics (log scale)", fontsize=13)
ax.set_xlabel("Epoch")
ax.set_ylabel("Prediction loss (log scale)")
ax.grid(True, which="both", axis="y", alpha=0.25)
ax.grid(True, which="major", axis="x", alpha=0.15)
ax.legend(title="Loss", fontsize=9)
plt.show()